# Knee OA KL-grade classification - v5 (from scratch) - Kaggle notebook

**Setup (once):** right panel -> *Add Input* -> search **"knee osteoarthritis dataset with severity"** (shashwatwork). Settings -> Accelerator -> **GPU T4 x2**.

| Cell | What it does | Time |
|---|---|---|
| 1 | **Training** (CleanLab downweighting, rotation pre-training, 12 models, ensemble) | ~5.5 h on a T4 |
| 2 | **Packaging**: model file, `predict.py`, CSV, plots, test-image galleries, zip | ~3 min |
| 3 | **Prints the final results clearly** + shows all the images | ~10 s |

* **Best way to run:** `Save Version -> Save & Run All (Commit)` with the GPU on. It keeps running with the browser closed.
* If you **already trained in this session**, skip cell 1 and run cells 2 and 3.
* Cell 1 first run: set `QUICK_TEST = True` at the top of cell 1 for a 3-5 minute check, then set it back to `False`.
* Download `knee_oa_final.zip` from the Output panel when done. Working files stay out of the output.


In [ ]:
# ==========================================================================================
# Knee OA KL-grade classification -- v5 (KAGGLE edition, from scratch, no pretrained weights)
#
# v4 result to beat (same 4,008-image test set, same 5 classes, nothing pretrained):
#   raw ensemble   acc 54.47%  weighted-F1 0.464  macro-F1 0.482  kappa 0.625
#   bias-tuned     acc 55.31%  weighted-F1 0.536  macro-F1 0.548  kappa 0.681   (G1 F1 0.213)
#
# WHAT v5 CHANGES (each change targets something visible in the v4 logs):
#   1. EMA of the weights (decay 0.998). v4 val kappa jumped +-0.1 between epochs, so "best epoch" was
#      partly luck. Checkpoints are now picked on the smoothed EMA model.
#   2. Deeper ResNet / DenseNet. v4's ResNet downsampled only 4x (DenseNet 8x) and had the weakest
#      receptive field; MixedKernelNet (16x) was the best model. ResNet now has 4 stages (/16, ~5M params),
#      DenseNet gets a 3rd dense block (/16).
#   3. Ordinal soft labels instead of uniform label smoothing: a G2 image puts its smoothing mass on
#      G1/G3 first, not on G4. Focal weight now uses the true-class probability.
#   4. Stronger regularisation (train loss 0.57 vs val loss 1.13 in v4 = overfitting): gamma/contrast
#      jitter, light noise, random erasing, and real AdamW weight decay (no decay on BN/bias).
#   5. 3-epoch LR warm-up + per-step cosine (v4 sat at kappa 0 for ~10 epochs in the CleanLab probes),
#      80 fine-tune epochs instead of 50, 4 seeds per architecture (12 models).
#   6. Extra decoder: expected-grade thresholds tuned on val (reported next to raw and bias-tuned).
#   Not changed on purpose: 5 classes, from scratch, test set, CleanLab downweighting, rotation SSL.
#
# HOW TO RUN ON KAGGLE
#   1. New notebook -> "Add Input" -> search "knee osteoarthritis dataset with severity" (shashwatwork).
#   2. Settings -> Accelerator: GPU T4 x2 (or P100). Internet ON is only needed if cleanlab/onnx are
#      missing (the script installs them itself; without internet it falls back safely).
#   3. Paste this WHOLE file into ONE cell. Set QUICK_TEST = True first (about 3-5 minutes) to check
#      everything works, then set it back to False.
#   4. For the real run use  Save Version -> "Save & Run All (Commit)"  with the GPU on. A committed run
#      keeps going with the browser closed and keeps /kaggle/working as the notebook Output.
#      (An interactive session loses /kaggle/working files when it ends.)
#
# Kaggle-specific changes versus the local script:
#   * dataset folder is auto-detected under /kaggle/input; artifacts go to /kaggle/working/artifacts_v5
#   * T4/P100 have no bf16 -> uses fp16 autocast + GradScaler (checked by a self-test; falls back to fp32)
#   * one log line for EVERY epoch + a per-batch tqdm bar + the OVERALL bar with ETA
#   * models are trained seed-major (c,d,e seed1 -> c,d,e seed2 -> ...) and a TIME_BUDGET_H guard stops
#     cleanly before Kaggle's 12 h limit, then reports with the models finished so far
#   * cleanlab / onnx are auto-installed; if there is no internet a built-in fallback is used
#   * result plots are displayed inline at the end
#
# What is different from v3 (Kaggle):
#   BUG FIX   ColorJitter used to run on already-normalised [-1,1] pixels and clamped every pixel
#             darker than mid-gray to a constant (training images only). Augmentation now runs in
#             [0,1] space, on the GPU, and normalisation happens afterwards.
#   HYGIENE   ResBlock: add THEN ReLU. DenseNet: BN-ReLU in transitions + final BN-ReLU before GAP.
#             No conv bias before BatchNorm. BatchNorm1d in the classifier head. Kaiming init.
#   SPEED     whole dataset lives on the GPU as uint8; no DataLoader workers; bf16 autocast;
#             channels_last. (Windows-safe: no multiprocessing.)
#   SAFETY    every stage is saved to ART_DIR and skipped on re-run (resume after sleep/crash),
#             one global progress bar with ETA, ONNX export for all 9 models at the end.
#   REPORT    raw ensemble + per-class-bias-tuned ensemble (bias tuned on VAL only, then scored on
#             test), per-model test metrics, plots.
#
# Kept from v3: CleanLab OOF with DOWNWEIGHTING + per-class caps, rotation-prediction pretraining,
# focal loss, ramped ordinal term, kappa-based checkpointing, flip-TTA, 3 archs x 3 seeds.
# ==========================================================================================
import os, sys, json, math, copy, random, time, shutil, warnings, contextlib, subprocess, importlib
from pathlib import Path
from collections import Counter
warnings.filterwarnings("ignore")
try:
    sys.stdout.reconfigure(line_buffering=True)      # Kaggle commit logs show lines immediately
except Exception:
    pass


def ensure_pkg(import_name, pip_name=None):
    """Import a package; if missing try a quiet pip install; return True/False (never raises)."""
    try:
        importlib.import_module(import_name)
        return True
    except ImportError:
        pass
    try:
        print(f"installing {pip_name or import_name} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--retries", "1", "--timeout", "15",
                        pip_name or import_name], check=True, capture_output=True, timeout=240)
        importlib.invalidate_caches()
        importlib.import_module(import_name)
        return True
    except Exception as e:
        print(f"  could not install {pip_name or import_name} ({type(e).__name__}); a built-in fallback will be used.")
        return False


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from tqdm.auto import tqdm

# ============================== CONFIG (edit this block) ==================================
ON_KAGGLE = Path("/kaggle/working").exists()
DATA_DIR = None      # None = auto-detect. Or paste the exact folder that contains train/ test/ val/ auto_test/
DEFAULT_DATA = "/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity"
BASE = None          # filled in by find_dataset() -- do not edit
ART_DIR = Path("/kaggle/working/artifacts_v5") if ON_KAGGLE else Path("artifacts_v5")
RESUME_FROM = None   # optional: folder of a previous v5 run's artifacts_v5 (added as a Kaggle input) to continue from
REUSE_WEIGHTS = None # optional: path to the v4 run's sample_weights.npy (skips ~45 min of CleanLab and keeps the exact
                     # same downweighted images as v4, so v4 vs v5 differ only in the model/training changes)
QUICK_TEST = False   # True = tiny end-to-end smoke run (a few minutes). DO THIS FIRST.
ALLOW_CPU = False    # leave False: refuse to start a multi-hour run without a working GPU
TIME_BUDGET_H = 11.0 # stop starting new models when this many hours of this session are used (Kaggle limit 12 h)
CACHE_DATA = False   # True = also save the decoded images as a ~500 MB .npz (not needed on Kaggle)

NC = 5
IMG = 224
BS = 64              # lower to 32 if you ever hit "CUDA out of memory" on 8 GB
LR = 1e-3
WD = 2e-2            # AdamW decoupled decay (applied to conv/linear weights only, not BN/bias). v4 used 1e-4 = almost none
WARMUP_EPOCHS = 3    # linear LR warm-up, then cosine to 0 (stepped per batch)
EMA_DECAY = 0.998    # weight EMA; ~500 steps (~6 epochs) horizon. Checkpoints are chosen on the EMA model
SEED = 42
SPLIT_VAL = 0.15
OOF_FOLDS = 5
OOF_EPOCHS = 35
SSL_EPOCHS = 15
EP_FT = 80
SEEDS_PER_ARCH = 4      # 3 archs x 4 seeds = 12 models (set 3 to save ~1 h)
DOWNWEIGHT_FACTOR = 0.30
MAX_FLAG_FRAC_SMALL = 0.10
MAX_FLAG_FRAC_LARGE = 0.30
MIN_CLASS_FOR_FULL_CAP = 500
FOCAL_GAMMA = 2.0
ORD_SMOOTH = 0.15    # soft-label mass spread to NEIGHBOURING grades (exp(-|k-y|) weights), replaces uniform smoothing
AUG_ERASE_P = 0.25   # probability of one random-erasing box per training image (side 10-30%)
ORD_LAMBDA_MAX = 0.05
ORD_WARMUP = 15
USE_AMP = True       # mixed precision: bf16 on Ampere+, fp16+GradScaler on T4/P100. False = plain fp32 (slower)
TUNE_OBJECTIVE = "macro_f1"   # what the per-class bias is tuned for on VAL: "macro_f1" or "kappa"
QUICK_N_TRAIN, QUICK_N_TEST = 480, 480

if QUICK_TEST:
    OOF_FOLDS, OOF_EPOCHS, SSL_EPOCHS, EP_FT, SEEDS_PER_ARCH = 2, 1, 1, 2, 1
    ART_DIR = Path(str(ART_DIR) + "_quicktest")
# ==========================================================================================

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_MODE = None                              # decided (and self-tested) in preflight(): "bf16", "fp16" or None
CL = DEV.type == "cuda"                      # channels_last only helps on GPU
CLASS_NAMES = ["Healthy", "Doubtful", "Minimal", "Moderate", "Severe"]
V_OLD = {  # your earlier runs, for the comparison chart
    "v1": {"acc": 0.4613, "f1w": 0.3165, "kappa": 0.4160},
    "v2": {"acc": 0.3114, "f1w": 0.2851, "kappa": 0.4597},
    "v3": {"acc": 0.5100, "f1w": 0.4482, "kappa": 0.5794},
    "v4 raw": {"acc": 0.5447, "f1w": 0.4643, "kappa": 0.6250},
    "v4 tuned": {"acc": 0.5531, "f1w": 0.5361, "kappa": 0.6812},
}

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def amp_ctx():
    if AMP_MODE == "bf16":
        return torch.autocast(device_type=DEV.type, dtype=torch.bfloat16)
    if AMP_MODE == "fp16":
        return torch.autocast(device_type=DEV.type, dtype=torch.float16)
    return contextlib.nullcontext()


def make_scaler():
    """GradScaler is only needed for fp16 (bf16 has fp32's exponent range)."""
    if AMP_MODE != "fp16":
        return None
    try:
        return torch.amp.GradScaler("cuda")
    except Exception:
        return torch.cuda.amp.GradScaler()


def opt_step(loss, opt, params, scaler):
    """backward + gradient clipping (5.0) + optimizer step, with fp16 loss-scaling when needed."""
    if scaler is not None:
        scaler.scale(loss).backward()
        scaler.unscale_(opt)                                 # clip real gradients, not scaled ones
        torch.nn.utils.clip_grad_norm_(params, 5.0)
        scaler.step(opt)
        scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 5.0)
        opt.step()


def _amp_selftest(mode):
    """One tiny forward+backward+optimizer step in the given precision mode. True if it works and is finite."""
    global AMP_MODE
    old = AMP_MODE
    AMP_MODE = mode
    try:
        conv, lin = nn.Conv2d(3, 8, 3).to(DEV), nn.Linear(8, 5).to(DEV)
        params = list(conv.parameters()) + list(lin.parameters())
        opt = torch.optim.AdamW(params, lr=1e-3)
        sc = make_scaler()
        x = torch.randn(8, 3, 64, 64, device=DEV)
        with amp_ctx():
            out = lin(conv(x).mean((2, 3)))
        loss = F.cross_entropy(out.float(), torch.zeros(8, dtype=torch.long, device=DEV))
        opt.zero_grad(set_to_none=True)
        opt_step(loss, opt, params, sc)
        if DEV.type == "cuda":
            torch.cuda.synchronize()
        return bool(torch.isfinite(loss).item())
    except Exception as e:
        print(f"  precision self-test '{mode}' failed: {type(e).__name__}: {str(e)[:120]}")
        return False
    finally:
        AMP_MODE = old


# ----------------------------------- dataset location --------------------------------------
def _is_dataset_root(d):
    return all((d / s).is_dir() for s in ("train", "test", "val"))


def find_dataset():
    for c in ([Path(DATA_DIR)] if DATA_DIR else []) + [Path(DEFAULT_DATA)]:
        if _is_dataset_root(c):
            return c
    root = Path("/kaggle/input")
    if root.is_dir():
        for dirpath, dirnames, _ in os.walk(root):
            d = Path(dirpath)
            if len(d.relative_to(root).parts) > 5:
                dirnames[:] = []
                continue
            if _is_dataset_root(d):
                return d
        tops = sorted(p.name for p in root.iterdir())
        raise FileNotFoundError(
            "Could not find the dataset (a folder containing train/, test/ and val/) under /kaggle/input.\n"
            f"/kaggle/input currently contains: {tops}\n"
            "Fix: use 'Add Input' on the right and add 'Knee Osteoarthritis Dataset with Severity' (shashwatwork), "
            "or set DATA_DIR at the top of the script to the exact folder.")
    raise FileNotFoundError("Dataset not found. Set DATA_DIR at the top of the script to the folder that contains "
                            "train/ test/ val/ (and auto_test/).")


# ----------------------------------- preflight ---------------------------------------------
def preflight():
    global BASE, AMP_MODE
    print(f"torch {torch.__version__} | CUDA build {torch.version.cuda} | cuda available: {torch.cuda.is_available()}")
    AMP_MODE = None
    if DEV.type == "cuda":
        name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
        total = torch.cuda.get_device_properties(0).total_memory / 2**30
        print(f"GPU: {name} | compute capability {cap[0]}.{cap[1]} | {total:.1f} GB VRAM | GPUs visible: {torch.cuda.device_count()} (using GPU 0)")
        if not _amp_selftest(None):                          # proves the installed PyTorch really has kernels for this GPU
            raise RuntimeError(
                "GPU kernel test FAILED (see message above). On Kaggle: Settings -> Accelerator -> choose 'GPU T4 x2' "
                "(the P100 can be incompatible with newer PyTorch builds), then restart the session and run again.\n"
                "On a local RTX 50-series card install PyTorch with CUDA 12.8+.")
        print("GPU kernel test (fp32): OK")
        if USE_AMP:
            for mode in (["bf16"] if cap[0] >= 8 else []) + ["fp16"]:
                if _amp_selftest(mode):
                    AMP_MODE = mode
                    break
        print(f"mixed precision: {AMP_MODE or 'OFF (fp32)'}")
    elif not ALLOW_CPU:
        raise RuntimeError("No GPU found. On Kaggle: Settings -> Accelerator -> GPU T4 x2, then restart the session. "
                           "(Set ALLOW_CPU = True only for a tiny QUICK_TEST.)")
    else:
        print("WARNING: running on CPU (ALLOW_CPU=True) -- only sensible for QUICK_TEST.")
    BASE = find_dataset()
    for sp in ["train", "test", "val"]:
        for c in range(NC):
            if not (BASE / sp / str(c)).is_dir():
                have = sorted(p.name for p in (BASE / sp).iterdir())[:10]
                raise FileNotFoundError(f"Expected class folder {BASE / sp / str(c)} but {BASE / sp} contains: {have}")
    if not (BASE / "auto_test").is_dir():
        print("NOTE: no auto_test/ folder in this dataset copy -> test set = test + val only.")
    print(f"Dataset folder OK: {BASE}")
    ART_DIR.mkdir(parents=True, exist_ok=True)
    if RESUME_FROM and Path(RESUME_FROM).is_dir() and not QUICK_TEST:
        n = 0
        for f in Path(RESUME_FROM).iterdir():
            if f.is_file() and not (ART_DIR / f.name).exists():
                shutil.copy2(f, ART_DIR / f.name); n += 1
        print(f"RESUME_FROM: copied {n} saved files from {RESUME_FROM}")


# ------------------------------------- data ------------------------------------------------
def _resample():
    return Image.Resampling.BILINEAR if hasattr(Image, "Resampling") else Image.BILINEAR


def read_split(split):
    items = []
    for c in range(NC):
        files = sorted(p for p in (BASE / split / str(c)).glob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
        if QUICK_TEST:
            files = files[:120]                              # smoke test: don't read the whole dataset
        items += [(p, c) for p in files]
    xs, ys, bad = [], [], 0
    for p, c in tqdm(items, desc=f"loading {split:9s}", unit="img", leave=True):
        try:
            im = Image.open(p)
            if len(xs) < 20 and im.mode == "RGB":   # one-off sanity check: are the 3 channels identical?
                a = np.asarray(im)
                if np.abs(a[..., 0].astype(int) - a[..., 1].astype(int)).max() > 2:
                    print(f"WARNING: {p.name} has non-identical colour channels; converting to grayscale changes it.")
            im = im.convert("L").resize((IMG, IMG), _resample())
            xs.append(np.asarray(im, dtype=np.uint8))
            ys.append(c)
        except Exception as e:                               # one corrupt file must not kill a multi-hour run
            bad += 1
            print(f"  skipped unreadable image {p.name}: {type(e).__name__}")
    if not xs:
        raise RuntimeError(f"No images could be read from {BASE / split}")
    if bad:
        print(f"  {split}: skipped {bad} unreadable images")
    return np.stack(xs), np.array(ys, dtype=np.int64)


def load_data():
    cache = ART_DIR / "data_u8.npz"
    if CACHE_DATA and cache.exists() and not QUICK_TEST:
        z = np.load(cache)
        print(f"loaded cached data: train {z['train_x'].shape}, test {z['test_x'].shape}")
        return z["train_x"], z["train_y"], z["test_x"], z["test_y"]
    print("reading images from disk (about 2-3 minutes) ...")
    trx, try_ = read_split("train")
    parts = [read_split(s) for s in ("test", "auto_test", "val") if (BASE / s).is_dir()]   # same test composition as v3
    tex = np.concatenate([p[0] for p in parts]); tey = np.concatenate([p[1] for p in parts])
    if QUICK_TEST:
        from sklearn.model_selection import train_test_split
        if len(trx) > QUICK_N_TRAIN:
            _, trx, _, try_ = train_test_split(trx, try_, test_size=QUICK_N_TRAIN, stratify=try_, random_state=SEED)
        if len(tex) > QUICK_N_TEST:
            _, tex, _, tey = train_test_split(tex, tey, test_size=QUICK_N_TEST, stratify=tey, random_state=SEED)
    elif CACHE_DATA:
        np.savez(cache, train_x=trx, train_y=try_, test_x=tex, test_y=tey)
    print(f"train {trx.shape}, test {tex.shape}")
    return trx, try_, tex, tey


def to_gpu(a_u8):                      # (N,H,W) uint8 numpy -> (N,1,H,W) uint8 on device
    return torch.from_numpy(a_u8).to(DEV).unsqueeze(1)


# --------------------------------- GPU augmentation ----------------------------------------
def gpu_augment(x):
    """x: (B,1,H,W) float in [0,1]. Rotation +-15 deg, translation +-10%, zoom 0.85-1.15,
    horizontal flip p=0.5, gamma 0.74-1.35, brightness/contrast +-20%, noise up to 0.03, random erasing.
    Everything is done in [0,1] space (v3 clamped all dark pixels because it did this after normalisation)."""
    B, dev = x.size(0), x.device
    ang = (torch.rand(B, device=dev) * 2 - 1) * math.radians(15)
    sc = 0.85 + torch.rand(B, device=dev) * 0.30
    tx = (torch.rand(B, device=dev) * 2 - 1) * 0.2          # 10% of the width = 0.2 in [-1,1] coordinates
    ty = (torch.rand(B, device=dev) * 2 - 1) * 0.2
    flip = 1 - 2 * (torch.rand(B, device=dev) < 0.5).float()   # draw < 0.5 -> -1 (mirror), else +1
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0] = cos * flip / sc; theta[:, 0, 1] = -sin / sc; theta[:, 0, 2] = tx
    theta[:, 1, 0] = sin * flip / sc; theta[:, 1, 1] = cos / sc;  theta[:, 1, 2] = ty
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    g = torch.exp((torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * 0.3)      # gamma 0.74-1.35 (X-ray exposure differences)
    x = x.clamp_min(1e-4) ** g
    b = 0.8 + torch.rand(B, 1, 1, 1, device=dev) * 0.4
    x = (x * b).clamp(0, 1)
    c = 0.8 + torch.rand(B, 1, 1, 1, device=dev) * 0.4
    m = x.mean(dim=(1, 2, 3), keepdim=True)
    x = ((x - m) * c + m).clamp(0, 1)
    x = (x + torch.randn_like(x) * (torch.rand(B, 1, 1, 1, device=dev) * 0.03)).clamp(0, 1)   # light sensor noise
    if AUG_ERASE_P > 0:                                                    # random erasing: one box, side 10-30 %
        H, W = x.shape[-2:]
        eh = (0.10 + 0.20 * torch.rand(B, device=dev)) * H
        ew = (0.10 + 0.20 * torch.rand(B, device=dev)) * W
        cy, cx = torch.rand(B, device=dev) * H, torch.rand(B, device=dev) * W
        ys = torch.arange(H, device=dev).view(1, H, 1).float()
        xs = torch.arange(W, device=dev).view(1, 1, W).float()
        box = ((ys - cy.view(B, 1, 1)).abs() < eh.view(B, 1, 1) / 2) & ((xs - cx.view(B, 1, 1)).abs() < ew.view(B, 1, 1) / 2)
        box = box & (torch.rand(B, device=dev) < AUG_ERASE_P).view(B, 1, 1)
        x = torch.where(box.unsqueeze(1), x.mean(dim=(1, 2, 3), keepdim=True).expand_as(x), x)
    return x


def prep_batch(xu8, augment):
    x = xu8.float().div(255.0)
    if augment:
        x = gpu_augment(x)
    x = (x - 0.5) / 0.5                                      # normalise AFTER augmentation
    x = x.expand(-1, 3, -1, -1)
    return x.contiguous(memory_format=torch.channels_last) if CL else x


def iterate(X, y, w, bs, shuffle):
    n = X.size(0)
    idx = torch.randperm(n, device=X.device) if shuffle else torch.arange(n, device=X.device)
    for i in range(0, n, bs):
        j = idx[i:i + bs]
        if shuffle and j.numel() < 8:                        # BatchNorm needs more than a couple of samples
            continue
        yield X[j], y[j], (w[j] if w is not None else None)


# ------------------------------------- models ----------------------------------------------
def make_head(in_dim):
    return nn.Sequential(nn.Linear(in_dim, 256, bias=False), nn.BatchNorm1d(256), nn.ReLU(),
                         nn.Dropout(0.4), nn.Linear(256, NC))


class MixedKernelNet(nn.Module):
    FEAT_DIM = 256
    def __init__(self):
        super().__init__()
        def blk(ci, co, k):
            return nn.Sequential(nn.Conv2d(ci, co, k, padding=k // 2, bias=False), nn.BatchNorm2d(co),
                                 nn.ReLU(), nn.MaxPool2d(2))
        self.block1, self.block2 = blk(3, 64, 3), blk(64, 128, 5)
        self.block3, self.block4 = blk(128, 128, 3), blk(128, 256, 5)
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.classifier = make_head(256)
    def extract_features(self, x):
        return self.pool(self.block4(self.block3(self.block2(self.block1(x)))))
    def forward(self, x):
        return self.classifier(self.extract_features(x))


class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(c)
    def forward(self, x):
        h = F.relu(self.bn1(self.conv1(x)))
        h = self.bn2(self.conv2(h))
        return F.relu(h + x)                                 # add THEN activate (standard residual block)


def down(ci, co):                                            # stride-2 3x3 conv-BN-ReLU (downsampling between stages)
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1, stride=2, bias=False), nn.BatchNorm2d(co), nn.ReLU())


class ResNet(nn.Module):
    """v5: four stages (64/128/256/256 channels, overall /16, ~5 M params). v4 had two stages and only /4."""
    FEAT_DIM = 256
    def __init__(self):
        super().__init__()
        self.entry = nn.Sequential(nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.stage1 = nn.Sequential(ResBlock(64), ResBlock(64))                              # 112x112
        self.stage2 = nn.Sequential(down(64, 128), ResBlock(128), ResBlock(128))             # 56x56
        self.stage3 = nn.Sequential(down(128, 256), ResBlock(256), ResBlock(256))           # 28x28
        self.stage4 = nn.Sequential(down(256, 256), ResBlock(256))                           # 14x14
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.classifier = make_head(256)
    def extract_features(self, x):
        return self.pool(self.stage4(self.stage3(self.stage2(self.stage1(self.entry(x))))))
    def forward(self, x):
        return self.classifier(self.extract_features(x))


class DenseBlock(nn.Module):
    def __init__(self, in_c, growth=16, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.Sequential(nn.BatchNorm2d(in_c), nn.ReLU(), nn.Conv2d(in_c, growth, 3, padding=1, bias=False)))
            in_c += growth
        self.out_c = in_c
    def forward(self, x):
        for layer in self.layers:
            x = torch.cat([x, layer(x)], dim=1)
        return x


def transition(cin, cout):
    return nn.Sequential(nn.BatchNorm2d(cin), nn.ReLU(), nn.Conv2d(cin, cout, 1, bias=False), nn.AvgPool2d(2))


class DenseNet(nn.Module):
    FEAT_DIM = 192
    def __init__(self):
        super().__init__()
        self.entry = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.db1 = DenseBlock(32); self.t1 = transition(self.db1.out_c, 64)      # 112 -> 56
        self.db2 = DenseBlock(64); self.t2 = transition(self.db2.out_c, 128)     # 56 -> 28
        self.db3 = DenseBlock(128); self.t3 = transition(self.db3.out_c, 192)    # 28 -> 14   (v5: third dense block)
        self.final = nn.Sequential(nn.BatchNorm2d(192), nn.ReLU())
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.classifier = make_head(192)
    def extract_features(self, x):
        x = self.t3(self.db3(self.t2(self.db2(self.t1(self.db1(self.entry(x)))))))
        return self.pool(self.final(x))
    def forward(self, x):
        return self.classifier(self.extract_features(x))


MODELS = {"c": MixedKernelNet, "d": ResNet, "e": DenseNet}
ARCH_NAMES = {"c": "MixedKernelNet", "d": "ResNet", "e": "DenseNet"}


def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)


def build_model(cls):
    m = cls()
    m.apply(init_weights)
    m.classifier[-1].reset_parameters()                      # output layer keeps PyTorch's default (small) init
    return m


def to_dev(m):
    m = m.to(DEV)
    return m.to(memory_format=torch.channels_last) if CL else m


# ------------------------------------- losses / predict ------------------------------------
def soft_targets(y):
    """(B,) grades -> (B,NC) soft labels: 1-ORD_SMOOTH on the true grade, ORD_SMOOTH spread over the OTHER grades
    with weights exp(-|k-y|), so the neighbouring grades get most of it."""
    ks = torch.arange(NC, device=y.device, dtype=torch.float32).view(1, -1)
    d = (ks - y.float().view(-1, 1)).abs()
    nb = torch.exp(-d) * (d > 0).float()
    nb = nb / nb.sum(1, keepdim=True)
    return (1 - ORD_SMOOTH) * F.one_hot(y, NC).float() + ORD_SMOOTH * nb


def focal_loss(logits, targets, class_w):
    logp = F.log_softmax(logits, dim=1)
    ce = -(soft_targets(targets) * logp).sum(1)                       # cross-entropy against the soft labels
    pt = logp.gather(1, targets.view(-1, 1)).squeeze(1).exp()        # probability given to the TRUE grade
    out = ((1 - pt).clamp_min(0) ** FOCAL_GAMMA) * ce
    return out * class_w[targets] if class_w is not None else out


@torch.no_grad()
def predict_probs(model, Xu8, tta, bs=256):
    model.eval()
    outs = []
    for i in range(0, Xu8.size(0), bs):
        xb = Xu8[i:i + bs]
        with amp_ctx():
            p = model(prep_batch(xb, False)).float().softmax(1)
            if tta:
                p = (p + model(prep_batch(xb.flip(-1), False)).float().softmax(1)) / 2
        outs.append(p.cpu())
    return torch.cat(outs).numpy()


# ------------------------------------ SSL pretraining --------------------------------------
def pretrain_rotation(cls, Xu8, epochs, tag, master):
    model = to_dev(build_model(cls))
    rot_head = nn.Linear(cls.FEAT_DIM, 4).to(DEV)
    opt = make_opt([model, rot_head])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = make_scaler()
    model_params = list(model.parameters())
    n = Xu8.size(0)
    n_batches = math.ceil(n / BS)
    for epoch in range(1, epochs + 1):
        t_ep = time.time()
        model.train()
        loss_sum, correct, seen = torch.zeros((), device=DEV), torch.zeros((), device=DEV), 0
        idx = torch.randperm(n, device=DEV)
        bar = tqdm(total=n_batches, desc=f"[{tag}-SSL] ep {epoch:2d}/{epochs}", unit="batch", leave=False,
                   mininterval=2.0, dynamic_ncols=True)
        for i in range(0, n, BS):
            j = idx[i:i + BS]
            if j.numel() < 8:
                continue
            x = Xu8[j].float().div(255.0)
            k = torch.randint(0, 4, (x.size(0),), device=DEV)
            xr = x.clone()
            for kk in (1, 2, 3):
                m = k == kk
                if m.any():
                    xr[m] = torch.rot90(x[m], kk, dims=(2, 3))
            xr = ((xr - 0.5) / 0.5).expand(-1, 3, -1, -1)
            if CL:
                xr = xr.contiguous(memory_format=torch.channels_last)
            opt.zero_grad(set_to_none=True)
            with amp_ctx():
                logits = rot_head(model.extract_features(xr))
            logits = logits.float()
            loss = F.cross_entropy(logits, k)
            opt_step(loss, opt, model_params, scaler)
            loss_sum += loss.detach() * j.numel(); correct += (logits.argmax(1) == k).sum(); seen += j.numel()
            bar.update(1)
        bar.close()
        sched.step()
        master.update(1)
        master.set_postfix_str(f"{tag} SSL ep{epoch}/{epochs}")
        tqdm.write(f"  [{tag}-SSL] ep {epoch:2d}/{epochs} | loss {loss_sum.item()/max(seen,1):.4f} | "
                   f"rotation acc {correct.item()/max(seen,1):.4f} | {time.time()-t_ep:5.1f}s")
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if not k.startswith("classifier")}


# ------------------------------------- supervised training ---------------------------------
def make_opt(modules):
    """AdamW with decoupled weight decay on conv/linear weights only (not on BatchNorm or biases)."""
    decay, no_decay = [], []
    for m in modules:
        for p in m.parameters():
            (decay if p.ndim > 1 else no_decay).append(p)
    return torch.optim.AdamW([{"params": decay, "weight_decay": WD}, {"params": no_decay, "weight_decay": 0.0}], lr=LR)


class WeightEMA:
    """Exponential moving average of ALL float tensors (weights + BatchNorm running stats). The EMA copy is what
    gets evaluated and checkpointed: it is much less noisy than the raw weights between epochs."""
    def __init__(self, model, decay):
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.decay, self.steps = decay, 0
        self.e = [t for t in self.model.state_dict().values() if t.is_floating_point()]
        self.m = [t for t in model.state_dict().values() if t.is_floating_point()]   # share storage with live params

    @torch.no_grad()
    def update(self):
        self.steps += 1
        d = min(self.decay, (1 + self.steps) / (10 + self.steps))                     # fast start, then EMA_DECAY
        torch._foreach_mul_(self.e, d)
        torch._foreach_add_(self.e, self.m, alpha=1 - d)


@torch.no_grad()
def evaluate(model, Xva, yva, yva_np):
    model.eval()
    vl, vb, preds = torch.zeros((), device=DEV), 0, []
    for xb, yb, _ in iterate(Xva, yva, None, 256, False):
        with amp_ctx():
            lo = model(prep_batch(xb, False))
        lo = lo.float()
        vl += F.cross_entropy(lo, yb); vb += 1
        preds.append(lo.argmax(1))
    preds = torch.cat(preds).cpu().numpy()
    kap = float(cohen_kappa_score(yva_np, preds, weights="quadratic"))
    return vl.item() / max(vb, 1), float(accuracy_score(yva_np, preds)), (kap if np.isfinite(kap) else 0.0)


def train_model(model, Xtr, ytr, wtr, Xva, yva, epochs, tag, class_w, master):
    model = to_dev(model)
    opt = make_opt([model])
    n_batches = math.ceil(Xtr.size(0) / BS)
    total_steps = epochs * n_batches
    warm = max(1, min(WARMUP_EPOCHS * n_batches, total_steps // 4))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda st: (st + 1) / warm if st < warm else 0.5 * (1 + math.cos(math.pi * (st - warm) / max(1, total_steps - warm))))
    ema = WeightEMA(model, EMA_DECAY)
    scaler = make_scaler()
    model_params = list(model.parameters())
    grades = torch.arange(NC, device=DEV, dtype=torch.float32)
    best_k, best_state = -1.0, None
    hist = {"train_loss": [], "val_loss": [], "val_acc": [], "val_kappa": [], "raw_val_kappa": []}
    yva_np = yva.cpu().numpy()
    t_start = time.time()
    for epoch in range(1, epochs + 1):
        t_ep = time.time()
        lr_now = opt.param_groups[0]["lr"]
        ord_l = ORD_LAMBDA_MAX * min(1.0, epoch / max(1, ORD_WARMUP))
        model.train()
        tl, nb = torch.zeros((), device=DEV), 0
        bar = tqdm(total=n_batches, desc=f"[{tag}] ep {epoch:2d}/{epochs}", unit="batch", leave=False,
                   mininterval=2.0, dynamic_ncols=True)
        for xb, yb, wb in iterate(Xtr, ytr, wtr, BS, True):
            wb = torch.ones(yb.size(0), device=DEV) if wb is None else wb
            xin = prep_batch(xb, True)                       # augmentation runs in fp32, outside autocast
            opt.zero_grad(set_to_none=True)
            with amp_ctx():
                logits = model(xin)
            logits = logits.float()
            per = focal_loss(logits, yb, class_w)
            ce = (per * wb).sum() / wb.sum().clamp_min(1e-8)
            exp_grade = (logits.softmax(1) * grades).sum(1)
            loss = ce + ord_l * F.mse_loss(exp_grade, yb.float())
            opt_step(loss, opt, model_params, scaler)
            sched.step()
            ema.update()
            tl += loss.detach(); nb += 1
            bar.update(1)
            if nb % 10 == 0:
                bar.set_postfix(loss=f"{tl.item() / nb:.4f}")
        bar.close()

        _, _, raw_kap = evaluate(model, Xva, yva, yva_np)                  # live weights (for information only)
        v_loss, v_acc, v_kap = evaluate(ema.model, Xva, yva, yva_np)       # EMA weights: used for checkpointing
        t_loss = tl.item() / max(nb, 1)
        hist["train_loss"].append(t_loss); hist["val_loss"].append(v_loss)
        hist["val_acc"].append(v_acc); hist["val_kappa"].append(v_kap); hist["raw_val_kappa"].append(raw_kap)
        star = ""
        if v_kap > best_k:
            best_k, star = v_kap, " *best*"
            best_state = {k: v.detach().cpu().clone() for k, v in ema.model.state_dict().items()}
        master.update(1)
        master.set_postfix_str(f"{tag} ep{epoch}/{epochs} ema_kappa {v_kap:.3f} best {best_k:.3f}")
        tqdm.write(f"  [{tag}] ep {epoch:2d}/{epochs} | train {t_loss:.4f} | val_loss {v_loss:.4f} | val_acc {v_acc:.4f} "
                   f"| kappa(EMA) {v_kap:.4f} raw {raw_kap:.4f} | best {best_k:.4f} | lr {lr_now:.1e} | "
                   f"{time.time() - t_ep:5.1f}s{star}")
    if best_state is None:                                   # cannot happen (best_k starts at -1), but never crash a long run
        best_state = {k: v.detach().cpu().clone() for k, v in ema.model.state_dict().items()}
    model.load_state_dict(best_state)                        # the returned model carries the best EMA weights
    tqdm.write(f"  [{tag}] done in {(time.time() - t_start) / 60:.1f} min | best val kappa (EMA) {best_k:.4f}")
    return model, hist, best_k, best_state


# ----------------------------------------- CleanLab ----------------------------------------
def fallback_label_issues(y, oof):
    """Used only if the cleanlab package cannot be installed (no internet). Same idea as confident learning:
    flag a sample when its given label is less likely than that class's average self-confidence AND the model's
    top prediction disagrees with the label; rank most-suspicious first."""
    self_conf = oof[np.arange(len(y)), y]
    thr = np.array([self_conf[y == c].mean() if (y == c).any() else 0.5 for c in range(NC)])
    flagged = np.where((self_conf < thr[y]) & (oof.argmax(1) != y))[0]
    return flagged[np.argsort(self_conf[flagged])]


def get_label_issues(y, oof):
    if ensure_pkg("cleanlab"):
        try:
            from cleanlab.filter import find_label_issues
            return find_label_issues(labels=y, pred_probs=oof, return_indices_ranked_by="self_confidence")
        except Exception as e:
            print(f"  cleanlab call failed ({type(e).__name__}: {str(e)[:100]}) -> using built-in fallback")
    return fallback_label_issues(y, oof)


def run_cleanlab(Xtr, ytr_t, master):
    y = ytr_t.cpu().numpy()
    skf = StratifiedKFold(n_splits=OOF_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros((len(y), NC), dtype=np.float32)
    for fold, (tr, va) in enumerate(skf.split(np.zeros(len(y)), y)):
        set_seed(SEED + 100 + fold)
        tr_t, va_t = torch.as_tensor(tr, device=DEV), torch.as_tensor(va, device=DEV)
        tqdm.write(f"\n  OOF fold {fold+1}/{OOF_FOLDS} ({len(tr)} train / {len(va)} val)")
        model, _, _, _ = train_model(build_model(ResNet), Xtr[tr_t], ytr_t[tr_t], None, Xtr[va_t], ytr_t[va_t],
                                     OOF_EPOCHS, f"OOF{fold+1}", None, master)
        oof[va] = predict_probs(model, Xtr[va_t], tta=False)
        del model
    ranked = get_label_issues(y, oof)
    counts = Counter(y.tolist())
    cap = {c: max(1, int(n * (MAX_FLAG_FRAC_SMALL if n < MIN_CLASS_FOR_FULL_CAP else MAX_FLAG_FRAC_LARGE)))
           for c, n in counts.items()}
    kept, used = [], Counter()
    for idx in ranked:                                       # most-likely-mislabelled first
        c = y[idx]
        if used[c] < cap[c]:
            kept.append(idx); used[c] += 1
    tqdm.write(f"\n  CleanLab raw flags: {len(ranked)}/{len(y)} ({len(ranked)/len(y):.1%}) | after per-class caps: "
               f"{len(kept)} ({len(kept)/len(y):.1%}) DOWNWEIGHTED to {DOWNWEIGHT_FACTOR}x, none removed | caps {cap}")
    sw = np.ones(len(y), dtype=np.float32)
    sw[np.array(kept, dtype=int)] = DOWNWEIGHT_FACTOR
    return sw, oof


# ----------------------------------------- reporting ---------------------------------------
def metrics_of(y, pred):
    return {"acc": float(accuracy_score(y, pred)), "f1w": float(f1_score(y, pred, average="weighted")),
            "f1m": float(f1_score(y, pred, average="macro")),
            "kappa": float(cohen_kappa_score(y, pred, weights="quadratic")),
            "f1_per_class": [float(v) for v in f1_score(y, pred, average=None, labels=list(range(NC)))],
            "cm": confusion_matrix(y, pred, labels=list(range(NC))).tolist()}


def show_metrics(title, m):
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
    print(f"Accuracy {m['acc']*100:.2f}% | weighted F1 {m['f1w']:.4f} | macro F1 {m['f1m']:.4f} | quad. kappa {m['kappa']:.4f}")
    print("Per-class F1: " + "  ".join(f"G{i} {f:.3f}" for i, f in enumerate(m["f1_per_class"])))
    cm = np.array(m["cm"])
    print("Confusion matrix (rows = true, cols = predicted):")
    for i in range(NC):
        print(f"  G{i}: [" + "  ".join(f"{cm[i, j]:4d}" for j in range(NC)) + "]")


def tune_bias(logp_val, y_val, objective):
    def score(b):
        pred = (logp_val + b).argmax(1)
        return (f1_score(y_val, pred, average="macro") if objective == "macro_f1"
                else cohen_kappa_score(y_val, pred, weights="quadratic"))
    b, best = np.zeros(NC), score(np.zeros(NC))
    for _ in range(3):
        for c in range(NC):
            for g in np.arange(-1.5, 1.51, 0.1):
                b2 = b.copy(); b2[c] = g
                s = score(b2)
                if s > best + 1e-9:
                    best, b = s, b2
    return b, best


def tune_thresholds(P_val, y_val, objective):
    """Alternative decoder: grade = number of thresholds the expected grade E=sum(k*p_k) exceeds. The 4 thresholds are
    tuned on VAL only (coordinate ascent), then applied unchanged to test."""
    E = (P_val * np.arange(NC)).sum(1)
    def score(th):
        pred = np.digitize(E, th)
        return (f1_score(y_val, pred, average="macro") if objective == "macro_f1"
                else cohen_kappa_score(y_val, pred, weights="quadratic"))
    th = np.array([0.5, 1.5, 2.5, 3.5]); best = score(th)
    for _ in range(3):
        for i in range(4):
            lo = th[i - 1] + 0.02 if i > 0 else -0.5
            hi = th[i + 1] - 0.02 if i < 3 else 4.5
            for g in np.arange(max(lo, th[i] - 0.6), min(hi, th[i] + 0.6) + 1e-9, 0.02):
                t2 = th.copy(); t2[i] = g
                sc = score(t2)
                if sc > best + 1e-9:
                    best, th = sc, t2
    return th, best


def plot_confusion(cm, title, fname):
    cm = np.array(cm); pct = cm / cm.sum(1, keepdims=True).clip(min=1) * 100
    fig, axes = plt.subplots(1, 2, figsize=(14, 6)); fig.suptitle(title)
    for ax, data, fmt, t in [(axes[0], cm, "d", "counts"), (axes[1], pct, ".1f", "row %")]:
        im = ax.imshow(data, cmap="Blues")
        ax.set_xticks(range(NC)); ax.set_yticks(range(NC))
        ax.set_xticklabels([f"G{i}" for i in range(NC)]); ax.set_yticklabels([f"G{i}\n{CLASS_NAMES[i]}" for i in range(NC)])
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(t)
        for i in range(NC):
            for j in range(NC):
                v = data[i, j]
                ax.text(j, i, f"{v:{fmt}}" + ("%" if fmt == ".1f" else ""), ha="center", va="center",
                        color="white" if v > data.max() / 2 else "black", fontsize=10)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.savefig(ART_DIR / fname, dpi=150, bbox_inches="tight"); plt.close()


def plot_per_class(y, pred, title, fname):
    p = precision_score(y, pred, average=None, labels=list(range(NC)), zero_division=0)
    r = recall_score(y, pred, average=None, labels=list(range(NC)), zero_division=0)
    f = f1_score(y, pred, average=None, labels=list(range(NC)), zero_division=0)
    x, w = np.arange(NC), 0.25
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for off, vals, lab, col in [(-w, p, "Precision", "#4472C4"), (0, r, "Recall", "#ED7D31"), (w, f, "F1", "#70AD47")]:
        bars = ax.bar(x + off, vals, w, label=lab, color=col)
        for b in bars:
            ax.annotate(f"{b.get_height():.2f}", (b.get_x() + b.get_width() / 2, b.get_height()), xytext=(0, 2),
                        textcoords="offset points", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([f"G{i}\n{CLASS_NAMES[i]}" for i in range(NC)])
    ax.set_ylim(0, 1.05); ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.savefig(ART_DIR / fname, dpi=150, bbox_inches="tight"); plt.close()


def plot_curves(hists):
    n = len(hists); cols = 3; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3.8 * rows)); axes = np.array(axes).reshape(-1)
    for ax, (tag, h) in zip(axes, sorted(hists.items())):
        ax.plot(h["train_loss"], label="train loss"); ax.plot(h["val_loss"], label="val loss")
        a2 = ax.twinx(); a2.plot(h["val_kappa"], color="purple", ls="--", label="val kappa")
        ax.set_title(f"{tag} ({ARCH_NAMES[tag[0]]})"); ax.set_xlabel("epoch"); ax.legend(loc="upper left", fontsize=7)
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.savefig(ART_DIR / "training_curves_v5.png", dpi=150); plt.close()


def plot_versions(raw, tuned):
    rows = dict(V_OLD); rows["v5 raw"] = raw; rows["v5 tuned"] = tuned
    keys = [("acc", "Accuracy"), ("f1w", "Weighted F1"), ("kappa", "Quad. kappa")]
    cols = ["#B00020", "#B8860B", "#4472C4", "#8FA9D6", "#5B7DB8", "#2E7D32", "#1B5E20"]
    x, w = np.arange(len(keys)), 0.12
    fig, ax = plt.subplots(figsize=(13, 5.5))
    for i, (name, m) in enumerate(rows.items()):
        bars = ax.bar(x + (i - 3) * w, [m[k] for k, _ in keys], w, label=name, color=cols[i])
        for b in bars:
            ax.annotate(f"{b.get_height():.3f}", (b.get_x() + b.get_width() / 2, b.get_height()), xytext=(0, 2),
                        textcoords="offset points", ha="center", fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels([n for _, n in keys]); ax.set_ylim(0, 1); ax.legend(); ax.set_title("All versions")
    plt.tight_layout(); plt.savefig(ART_DIR / "version_comparison_v5.png", dpi=150, bbox_inches="tight"); plt.close()


def plot_predictions(test_x, y, pred, probs):
    fig, axes = plt.subplots(NC, 3, figsize=(12, NC * 3.5)); rng = np.random.RandomState(SEED)
    fig.suptitle("Test samples: true vs predicted (v5 ensemble, raw)", y=1.01)
    for c in range(NC):
        idx = np.where(y == c)[0]
        pick = rng.choice(idx, min(3, len(idx)), replace=False) if len(idx) else []
        axes[c, 0].set_ylabel(f"G{c} {CLASS_NAMES[c]}", fontweight="bold")
        for j in range(3):
            ax = axes[c, j]; ax.axis("off")
            if j < len(pick):
                i = pick[j]
                ax.imshow(test_x[i], cmap="gray", vmin=0, vmax=255)
                ax.set_title(f"true G{y[i]} | pred G{pred[i]} | {probs[i, pred[i]]:.0%}", fontsize=9,
                             color="green" if pred[i] == y[i] else "red")
    plt.tight_layout(); plt.savefig(ART_DIR / "test_predictions_v5.png", dpi=150, bbox_inches="tight"); plt.close()


def export_all_onnx(tags):
    if not ensure_pkg("onnx"):
        print("onnx is not available -> skipping ONNX export (the .pt checkpoints in the artifacts folder are enough).")
        return
    for tag in tags:
        try:
            model = build_model(MODELS[tag[0]])
            model.load_state_dict(torch.load(ART_DIR / f"ft_{tag}.pt", map_location="cpu")["state"])
            model.eval().cpu()
            path = ART_DIR / f"model_{tag}_v5.onnx"
            dummy = torch.randn(1, 3, IMG, IMG)
            kw = dict(input_names=["input"], output_names=["output"], opset_version=13,
                      dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}})
            try:
                torch.onnx.export(model, dummy, str(path), dynamo=False, **kw)
            except TypeError:
                torch.onnx.export(model, dummy, str(path), **kw)
            print(f"  {path.name}: {path.stat().st_size / 2**20:.2f} MB")
        except Exception as e:
            print(f"  ONNX export failed for {tag}: {e}")


def show_images(names):
    """Display the saved PNGs inline when running in a notebook (silently skipped in a plain script)."""
    try:
        ip = sys.modules.get("IPython")
        if ip is None or ip.get_ipython() is None:
            return
        from IPython.display import display, Image as IPImage
        for n in names:
            p = ART_DIR / n
            if p.exists():
                print(f"\n--- {n} ---")
                display(IPImage(filename=str(p)))
    except Exception:
        pass


def report(test_x, test_y):
    """Everything below reads from saved artifacts, so it can be re-run any time without the GPU."""
    meta = json.load(open(ART_DIR / "models_index.json"))
    tags, kappas = meta["tags"], meta["val_kappa"]
    if not tags:
        print("No trained models found in the artifacts folder -> nothing to report.")
        return None, None
    y_val = np.load(ART_DIR / "val_labels.npy")
    print(f"\n{'=' * 60}\nPER-MODEL RESULTS (test numbers are for reporting only; nothing is selected on test)\n{'=' * 60}")
    for t in tags:
        pt = np.load(ART_DIR / f"probs_{t}_test.npy").argmax(1)
        print(f"  {t} ({ARCH_NAMES[t[0]]:14s}) best val kappa {kappas[t]:.4f} | test acc {accuracy_score(test_y, pt):.4f} "
              f"| test kappa {cohen_kappa_score(test_y, pt, weights='quadratic'):.4f}")
    w = np.clip(np.array([kappas[t] for t in tags]), 1e-3, None); w = w / w.sum()
    Pva = sum(wi * np.load(ART_DIR / f"probs_{t}_val.npy") for wi, t in zip(w, tags))
    Pte = sum(wi * np.load(ART_DIR / f"probs_{t}_test.npy") for wi, t in zip(w, tags))
    raw = metrics_of(test_y, Pte.argmax(1))
    show_metrics(f"ENSEMBLE of {len(tags)} models -- RAW (flip-TTA, kappa-weighted)", raw)

    bias, vbest = tune_bias(np.log(Pva + 1e-9), y_val, TUNE_OBJECTIVE)
    tuned_pred = (np.log(Pte + 1e-9) + bias).argmax(1)
    tuned = metrics_of(test_y, tuned_pred)
    print(f"\nPer-class bias tuned on VAL for {TUNE_OBJECTIVE}: {np.round(bias, 2).tolist()} (val score {vbest:.4f})")
    show_metrics("ENSEMBLE -- BIAS-TUNED (tuned on val, scored once on test)", tuned)

    th, tbest = tune_thresholds(Pva, y_val, TUNE_OBJECTIVE)
    thr_pred = np.digitize((Pte * np.arange(NC)).sum(1), th)
    thr = metrics_of(test_y, thr_pred)
    print(f"\nExpected-grade thresholds tuned on VAL for {TUNE_OBJECTIVE}: {np.round(th, 2).tolist()} (val score {tbest:.4f})")
    show_metrics("ENSEMBLE -- THRESHOLD-DECODED (tuned on val, scored once on test)", thr)

    print(f"\n{'=' * 60}\nv5 vs v4 (same 4,008 test images)\n{'=' * 60}")
    print(f"{'':14s}{'acc':>9s}{'wF1':>9s}{'kappa':>9s}")
    for name, m in [("v4 raw", V_OLD["v4 raw"]), ("v5 raw", raw), ("v4 tuned", V_OLD["v4 tuned"]), ("v5 tuned", tuned)]:
        print(f"{name:14s}{m['acc']*100:8.2f}%{m['f1w']:9.4f}{m['kappa']:9.4f}")
    print(f"{'v5 threshold':14s}{thr['acc']*100:8.2f}%{thr['f1w']:9.4f}{thr['kappa']:9.4f}")

    plot_confusion(raw["cm"], "Ensemble, raw", "confusion_matrix_raw_v5.png")
    plot_confusion(tuned["cm"], "Ensemble, bias-tuned", "confusion_matrix_tuned_v5.png")
    plot_per_class(test_y, Pte.argmax(1), "Per-class precision/recall/F1 -- raw", "per_class_raw_v5.png")
    plot_per_class(test_y, tuned_pred, "Per-class precision/recall/F1 -- bias-tuned", "per_class_tuned_v5.png")
    plot_versions(raw, tuned)
    plot_predictions(test_x, test_y, Pte.argmax(1), Pte)
    hists = {t: json.load(open(ART_DIR / f"hist_{t}.json")) for t in tags}
    plot_curves(hists)
    json.dump({"raw": raw, "tuned": tuned, "threshold": thr, "thresholds": th.tolist(), "bias": bias.tolist(), "tune_objective": TUNE_OBJECTIVE,
               "val_kappa": kappas}, open(ART_DIR / "metrics_v5.json", "w"), indent=2)
    print(f"\nPlots + metrics_v5.json saved in {ART_DIR.resolve()}")
    show_images(["confusion_matrix_raw_v5.png", "confusion_matrix_tuned_v5.png", "per_class_tuned_v5.png",
                 "version_comparison_v5.png", "test_predictions_v5.png", "training_curves_v5.png"])
    return raw, tuned


# ------------------------------------------ main -------------------------------------------
def main():
    t0 = time.time()
    if QUICK_TEST and ART_DIR.exists():
        shutil.rmtree(ART_DIR)
    print(f"{'QUICK TEST' if QUICK_TEST else 'FULL RUN'} | artifacts -> {ART_DIR}")
    preflight()
    set_seed(SEED)
    train_x, train_y, test_x, test_y = load_data()

    pos = np.arange(len(train_y))
    tr_pos, va_pos = next(StratifiedShuffleSplit(n_splits=1, test_size=SPLIT_VAL, random_state=SEED).split(pos, train_y))
    Xtr, ytr = to_gpu(train_x[tr_pos]), torch.as_tensor(train_y[tr_pos], device=DEV)
    Xva, yva = to_gpu(train_x[va_pos]), torch.as_tensor(train_y[va_pos], device=DEV)
    Xte = to_gpu(test_x)
    np.save(ART_DIR / "val_labels.npy", train_y[va_pos])
    counts = Counter(train_y[tr_pos].tolist())
    class_w = torch.tensor([len(tr_pos) / (NC * counts[i]) for i in range(NC)], dtype=torch.float32, device=DEV)
    print(f"train {len(tr_pos)} | val {len(va_pos)} | test {len(test_y)} | class counts {dict(sorted(counts.items()))}")
    print("class weights:", [round(float(v), 3) for v in class_w])
    print("parameters: " + " | ".join(f"{ARCH_NAMES[k]} {sum(p.numel() for p in build_model(c).parameters())/1e6:.2f} M"
                                   for k, c in sorted(MODELS.items())))

    total_epochs = OOF_FOLDS * OOF_EPOCHS + len(MODELS) * SSL_EPOCHS + len(MODELS) * SEEDS_PER_ARCH * EP_FT
    master = tqdm(total=total_epochs, desc="OVERALL", unit="epoch", dynamic_ncols=True)

    # ---- stage 1: CleanLab out-of-fold -> sample weights (downweight, never remove) ----
    sw_path = ART_DIR / "sample_weights.npy"
    if not sw_path.exists() and REUSE_WEIGHTS and Path(REUSE_WEIGHTS).is_file() and not QUICK_TEST:
        w_old = np.load(REUSE_WEIGHTS)
        if w_old.shape == (len(tr_pos),):
            np.save(sw_path, w_old.astype(np.float32))
            tqdm.write(f"[reuse] CleanLab weights copied from {REUSE_WEIGHTS} ({int((w_old < 1).sum())} downweighted)")
        else:
            tqdm.write(f"[reuse] {REUSE_WEIGHTS} has shape {w_old.shape}, expected ({len(tr_pos)},) -> running CleanLab instead")
    if sw_path.exists():
        sw = np.load(sw_path); master.update(OOF_FOLDS * OOF_EPOCHS)
        tqdm.write(f"[resume] CleanLab weights loaded ({int((sw < 1).sum())} downweighted)")
    else:
        sw, oof = run_cleanlab(Xtr, ytr, master)
        np.save(sw_path, sw); np.save(ART_DIR / "oof_probs.npy", oof)
    sw_t = torch.as_tensor(sw, device=DEV)

    # ---- stage 2: self-supervised rotation pretraining (train images only, no labels) ----
    ssl_states = {}
    for arch, cls in sorted(MODELS.items()):
        p = ART_DIR / f"ssl_{arch}.pt"
        if p.exists():
            ssl_states[arch] = torch.load(p, map_location="cpu"); master.update(SSL_EPOCHS)
            tqdm.write(f"[resume] SSL backbone {arch} loaded")
        else:
            set_seed(SEED + 7)
            ssl_states[arch] = pretrain_rotation(cls, Xtr, SSL_EPOCHS, arch, master)
            torch.save(ssl_states[arch], p)

    # ---- stage 3: fine-tune 3 archs x seeds on the FULL training split (seed-major order) ----
    # order c_s1, d_s1, e_s1, c_s2, ... so that if the time budget ever cuts the run short, the ensemble that
    # exists is still balanced across the three architectures.
    tags, kappas, model_hours = [], {}, []
    out_of_time = False
    for s in range(SEEDS_PER_ARCH):
        for arch, cls in sorted(MODELS.items()):
            tag = f"{arch}_s{s+1}"
            ck = ART_DIR / f"ft_{tag}.pt"
            if ck.exists() and (ART_DIR / f"probs_{tag}_test.npy").exists():
                kappas[tag] = float(torch.load(ck, map_location="cpu")["val_kappa"]); master.update(EP_FT)
                tags.append(tag)
                tqdm.write(f"[resume] {tag} already trained (val kappa {kappas[tag]:.4f})")
                continue
            used_h = (time.time() - t0) / 3600
            need_h = 1.15 * max(model_hours) if model_hours else 0.0
            if used_h + need_h > TIME_BUDGET_H:
                tqdm.write(f"\n[TIME BUDGET] {used_h:.2f} h used, next model needs ~{need_h:.2f} h, budget {TIME_BUDGET_H} h "
                           f"-> stopping here and reporting with the {len(tags)} finished models. "
                           f"Re-run (or use RESUME_FROM) to train the rest.")
                out_of_time = True
                break
            t_model = time.time()
            set_seed(SEED + s + 1)
            model = build_model(cls)
            missing, unexpected = model.load_state_dict(ssl_states[arch], strict=False)
            assert not unexpected and all(k.startswith("classifier") for k in missing), (missing, unexpected)
            tqdm.write(f"\n  training {tag} ({ARCH_NAMES[arch]}, seed {s+1}) -- model {len(tags)+1}/{len(MODELS)*SEEDS_PER_ARCH}")
            model, hist, best_k, best_state = train_model(model, Xtr, ytr, sw_t, Xva, yva, EP_FT, tag, class_w, master)
            np.save(ART_DIR / f"probs_{tag}_val.npy", predict_probs(model, Xva, tta=True))
            np.save(ART_DIR / f"probs_{tag}_test.npy", predict_probs(model, Xte, tta=True))
            json.dump(hist, open(ART_DIR / f"hist_{tag}.json", "w"))
            torch.save({"state": best_state, "val_kappa": best_k}, ck)
            kappas[tag] = best_k
            tags.append(tag)
            model_hours.append((time.time() - t_model) / 3600)
            del model
            if DEV.type == "cuda":
                torch.cuda.empty_cache()
            json.dump({"tags": tags, "val_kappa": kappas}, open(ART_DIR / "models_index.json", "w"))
            tqdm.write(f"  [{tag}] saved. elapsed {(time.time()-t0)/3600:.2f} h")
        if out_of_time:
            break
    master.close()
    json.dump({"tags": tags, "val_kappa": kappas}, open(ART_DIR / "models_index.json", "w"))

    # ---- stage 4: report + export (reads only saved artifacts) ----
    if not tags:
        print("No model was trained (time budget too small?). Nothing to report.")
        return
    report(test_x, test_y)
    print("\nONNX export (all models):")
    export_all_onnx(tags)
    print(f"\nTOTAL WALL TIME: {(time.time() - t0) / 3600:.2f} h")
    print("Reminder: 89-95% is not reachable from scratch on this dataset; published pretrained ensembles on larger data reach ~75-77%.")


if __name__ == "__main__":
    main()


In [ ]:
# ==========================================================================================
# Knee OA v5 -- FINALIZE: turns the finished training run into a small, clean set of deliverables.
# NO retraining. Paste into ONE new cell AFTER the training run (same session), or in a notebook that has the
# dataset and the run's output (artifacts_v5) as inputs.  CPU or GPU, about 2-4 minutes.
#
# Everything lands in /kaggle/working/final/ (and one zip next to it):
#   knee_oa_model.pt            the selected ensemble in ONE file (weights + bias + class names)
#   predict.py                  laptop script: python predict.py xray.png  -> grade, probabilities, Grad-CAM
#   RESULTS.md / results.json   final numbers on 4,008 test images AND the official 1,656-image test split
#   predictions_all_test.csv    one row per test image (file, true grade, prediction, probabilities)
#   confusion_matrix.png  per_class_f1.png  version_comparison.png  training_curves.png
#   test_gallery.png  gallery_mistakes.png  gradcam_examples.png  notebook_test_images.png  extra_5_test_images.png
#   demo_images/                25 test X-rays, ONE png each: Input X-ray | Model output (Grad-CAM, grade, confidence) | Ground truth
#   backup/all_model_weights.zip  every trained model (so nothing is lost)
# The many working files of the training run are then moved to /kaggle/temp (not part of the notebook output).
# ==========================================================================================
import os, sys, json, shutil, zipfile, importlib.util, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, precision_score, recall_score

# ------------------------------------ settings -------------------------------------------------
ART_DIR = None            # None = auto-find artifacts_v5 (must contain models_index.json)
DATA_DIR = None           # None = auto-find the dataset under /kaggle/input
DEFAULT_DATA = "/kaggle/input/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity"
OUT_DIR = Path("/kaggle/working/final") if Path("/kaggle/working").exists() else Path("final")
IMG = 224
TUNE_OBJECTIVE = "macro_f1"
MIN_MODELS = 3            # the selected ensemble always has at least this many models
MOVE_WORK_TO_TEMP = True  # move the training run's working files out of the notebook output when done
SEED = 7
NC = 5
CLASS_NAMES = ["Healthy", "Doubtful", "Minimal", "Moderate", "Severe"]
EXT = {".png", ".jpg", ".jpeg"}
V_OLD = {"v1": {"acc": 0.4613, "f1w": 0.3165, "kappa": 0.4160}, "v2": {"acc": 0.3114, "f1w": 0.2851, "kappa": 0.4597},
         "v3": {"acc": 0.5100, "f1w": 0.4482, "kappa": 0.5794}, "v4 raw": {"acc": 0.5447, "f1w": 0.4643, "kappa": 0.6250},
         "v4 tuned": {"acc": 0.5531, "f1w": 0.5361, "kappa": 0.6812}}

# ------------------------------- model code (identical to training) ------------------------------
MODEL_DEFS = r'''def make_head(in_dim):
    return nn.Sequential(nn.Linear(in_dim, 256, bias=False), nn.BatchNorm1d(256), nn.ReLU(),
                         nn.Dropout(0.4), nn.Linear(256, NC))


class MixedKernelNet(nn.Module):
    FEAT_DIM = 256
    def __init__(self):
        super().__init__()
        def blk(ci, co, k):
            return nn.Sequential(nn.Conv2d(ci, co, k, padding=k // 2, bias=False), nn.BatchNorm2d(co),
                                 nn.ReLU(), nn.MaxPool2d(2))
        self.block1, self.block2 = blk(3, 64, 3), blk(64, 128, 5)
        self.block3, self.block4 = blk(128, 128, 3), blk(128, 256, 5)
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.classifier = make_head(256)
    def extract_features(self, x):
        return self.pool(self.block4(self.block3(self.block2(self.block1(x)))))
    def forward(self, x):
        return self.classifier(self.extract_features(x))


class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(c)
    def forward(self, x):
        h = F.relu(self.bn1(self.conv1(x)))
        h = self.bn2(self.conv2(h))
        return F.relu(h + x)                                 # add THEN activate (standard residual block)


def down(ci, co):                                            # stride-2 3x3 conv-BN-ReLU (downsampling between stages)
    return nn.Sequential(nn.Conv2d(ci, co, 3, padding=1, stride=2, bias=False), nn.BatchNorm2d(co), nn.ReLU())


class ResNet(nn.Module):
    """v5: four stages (64/128/256/256 channels, overall /16, ~5 M params). v4 had two stages and only /4."""
    FEAT_DIM = 256
    def __init__(self):
        super().__init__()
        self.entry = nn.Sequential(nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.stage1 = nn.Sequential(ResBlock(64), ResBlock(64))                              # 112x112
        self.stage2 = nn.Sequential(down(64, 128), ResBlock(128), ResBlock(128))             # 56x56
        self.stage3 = nn.Sequential(down(128, 256), ResBlock(256), ResBlock(256))           # 28x28
        self.stage4 = nn.Sequential(down(256, 256), ResBlock(256))                           # 14x14
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.classifier = make_head(256)
    def extract_features(self, x):
        return self.pool(self.stage4(self.stage3(self.stage2(self.stage1(self.entry(x))))))
    def forward(self, x):
        return self.classifier(self.extract_features(x))


class DenseBlock(nn.Module):
    def __init__(self, in_c, growth=16, n_layers=3):
        super().__init__()
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.Sequential(nn.BatchNorm2d(in_c), nn.ReLU(), nn.Conv2d(in_c, growth, 3, padding=1, bias=False)))
            in_c += growth
        self.out_c = in_c
    def forward(self, x):
        for layer in self.layers:
            x = torch.cat([x, layer(x)], dim=1)
        return x


def transition(cin, cout):
    return nn.Sequential(nn.BatchNorm2d(cin), nn.ReLU(), nn.Conv2d(cin, cout, 1, bias=False), nn.AvgPool2d(2))


class DenseNet(nn.Module):
    FEAT_DIM = 192
    def __init__(self):
        super().__init__()
        self.entry = nn.Sequential(nn.Conv2d(3, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.db1 = DenseBlock(32); self.t1 = transition(self.db1.out_c, 64)      # 112 -> 56
        self.db2 = DenseBlock(64); self.t2 = transition(self.db2.out_c, 128)     # 56 -> 28
        self.db3 = DenseBlock(128); self.t3 = transition(self.db3.out_c, 192)    # 28 -> 14   (v5: third dense block)
        self.final = nn.Sequential(nn.BatchNorm2d(192), nn.ReLU())
        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.classifier = make_head(192)
    def extract_features(self, x):
        x = self.t3(self.db3(self.t2(self.db2(self.t1(self.db1(self.entry(x)))))))
        return self.pool(self.final(x))
    def forward(self, x):
        return self.classifier(self.extract_features(x))


MODELS = {"c": MixedKernelNet, "d": ResNet, "e": DenseNet}
ARCH_NAMES = {"c": "MixedKernelNet", "d": "ResNet", "e": "DenseNet"}
'''
PREDICT_SRC = r'''#!/usr/bin/env python
"""Knee osteoarthritis KL-grade predictor (from-scratch ensemble, no pretrained weights).

Usage:   python predict.py knee_xray.png [more.png ...] [--bundle knee_oa_model.pt] [--out-dir predictions]
Needs:   pip install torch pillow numpy matplotlib      (CPU is fine; a GPU is used automatically if present)
Output:  grade 0-4 + probability of every grade printed in the terminal, and a PNG per image with the X-ray,
         a Grad-CAM heat-map (where the model looked) and the probability bars.
Research prototype for a course project. NOT a medical device.
"""
import argparse, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

NC = 5
CLASS_NAMES = ["Healthy", "Doubtful", "Minimal", "Moderate", "Severe"]

@@MODEL_DEFS@@


def load_bundle(path, device):
    b = torch.load(path, map_location="cpu", weights_only=True)
    models = []
    for m in b["models"]:
        net = MODELS[m["arch"]]()
        net.load_state_dict(m["state"])
        models.append((net.eval().to(device), float(m["weight"]), m["tag"]))
    tot = sum(w for _, w, _ in models)
    b["_models"] = [(n, w / tot, t) for n, w, t in models]
    b["_device"] = device
    return b


def preprocess(path, size):
    im = Image.open(path).convert("L").resize((size, size), Image.BILINEAR)
    arr = np.asarray(im, dtype=np.uint8)
    x = (torch.from_numpy(arr.astype(np.float32)) / 255.0 - 0.5) / 0.5
    return x.view(1, 1, size, size).expand(1, 3, -1, -1).contiguous(), arr


@torch.no_grad()
def ensemble_probs(b, x):
    x = x.to(b["_device"])
    P = 0
    for net, w, _ in b["_models"]:
        p = (net(x).softmax(1) + net(x.flip(-1)).softmax(1)) / 2          # horizontal-flip TTA
        P = P + w * p
    return P[0].cpu().numpy()


def final_grade(b, P):
    raw = int(P.argmax())
    adj = int((np.log(P + 1e-9) + np.array(b["bias"])).argmax())          # per-class bias tuned on validation
    return raw, adj


def grad_cam(b, x, cls):
    """Grad-CAM of the highest-weight model for class `cls`. Returns a (H,W) map in [0,1]."""
    net = max(b["_models"], key=lambda t: t[1])[0]
    layer = getattr(net, {"MixedKernelNet": "block4", "ResNet": "stage4", "DenseNet": "final"}[type(net).__name__])
    store = {}
    def fwd(m, i, o):
        store["a"] = o
        o.register_hook(lambda g: store.__setitem__("g", g))
    h = layer.register_forward_hook(fwd)
    try:
        with torch.enable_grad():
            out = net(x.to(b["_device"]))
            net.zero_grad()
            out[0, cls].backward()
    finally:
        h.remove()
    a, g = store["a"].detach(), store["g"]
    raw = (g.mean((2, 3), keepdim=True) * a).sum(1, keepdim=True)
    cam = F.relu(raw)
    if float(cam.max()) <= 1e-12:                                          # degenerate map: fall back to the signed map
        cam = raw
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
    cam = cam - cam.min()
    return cam / cam.max() if cam.max() > 0 else cam


def predict_image(b, path, out_png=None, cam=True, size=None, true=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    size = size or b["img"]
    x, arr = preprocess(path, size)
    P = ensemble_probs(b, x)
    raw, grade = final_grade(b, P)
    heat = grad_cam(b, x, grade) if cam else None
    if out_png and true is not None:                                        # comparison layout: input | model output | ground truth
        t = int(true); ok = (t == grade)
        fig, ax = plt.subplots(1, 3, figsize=(15, 5.6))
        ax[0].imshow(arr, cmap="gray"); ax[0].set_title("Input X-ray", fontsize=12); ax[0].axis("off")
        ax[1].imshow(arr, cmap="gray")
        if cam:
            ax[1].imshow(heat, cmap="jet", alpha=0.40)
        ax[1].set_title(f"Model output: G{grade} {CLASS_NAMES[grade]}\nconfidence {P[grade]:.0%}" + ("  (Grad-CAM)" if cam else ""),
                        fontsize=12, color="#2E7D32" if ok else "#C62828"); ax[1].axis("off")
        ax[1].text(0.5, -0.04, "  ".join(f"G{i} {P[i]:.0%}" for i in range(NC)), transform=ax[1].transAxes,
                   ha="center", va="top", fontsize=9, color="#444")
        ax[2].imshow(arr, cmap="gray"); ax[2].set_title(f"Ground truth: G{t} {CLASS_NAMES[t]}", fontsize=12); ax[2].axis("off")
        fig.suptitle(("CORRECT" if ok else "WRONG") + f"   (true G{t}, predicted G{grade})", fontsize=13, fontweight="bold",
                     color="#2E7D32" if ok else "#C62828")
        plt.tight_layout(rect=(0, 0.02, 1, 0.94)); plt.savefig(out_png, dpi=130, bbox_inches="tight"); plt.close(fig)
    elif out_png:
        fig, ax = plt.subplots(1, 3 if cam else 2, figsize=(13 if cam else 9, 4.2))
        ax[0].imshow(arr, cmap="gray"); ax[0].set_title("input (224x224 grayscale)"); ax[0].axis("off")
        k = 1
        if cam:
            ax[1].imshow(arr, cmap="gray"); ax[1].imshow(heat, cmap="jet", alpha=0.40)
            ax[1].set_title(f"Grad-CAM (class G{grade})"); ax[1].axis("off"); k = 2
        cols = ["#2E7D32" if i == grade else "#9AA5B1" for i in range(NC)]
        ax[k].barh(range(NC), P, color=cols); ax[k].set_yticks(range(NC))
        ax[k].set_yticklabels([f"G{i} {CLASS_NAMES[i]}" for i in range(NC)]); ax[k].invert_yaxis(); ax[k].set_xlim(0, 1)
        ax[k].set_title(f"prediction: G{grade} {CLASS_NAMES[grade]}  ({P[grade]:.0%})")
        plt.tight_layout(); plt.savefig(out_png, dpi=130, bbox_inches="tight"); plt.close(fig)
    return {"grade": grade, "raw_grade": raw, "probs": P, "confidence": float(P[grade]), "cam": heat, "img": arr}


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("images", nargs="+")
    ap.add_argument("--bundle", default=str(Path(__file__).with_name("knee_oa_model.pt")))
    ap.add_argument("--out-dir", default="predictions")
    ap.add_argument("--no-cam", action="store_true")
    a = ap.parse_args()
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    b = load_bundle(a.bundle, dev)
    print(f"loaded {len(b['_models'])} models on {dev}")
    out = Path(a.out_dir); out.mkdir(parents=True, exist_ok=True)
    for p in a.images:
        r = predict_image(b, p, out / (Path(p).stem + "_pred.png"), cam=not a.no_cam)
        print(f"\n{p}\n  predicted KL grade: G{r['grade']} ({CLASS_NAMES[r['grade']]})  confidence {r['confidence']:.1%}"
              f"   [plain argmax without bias: G{r['raw_grade']}]")
        for i in range(NC):
            print(f"    G{i} {CLASS_NAMES[i]:9s} {r['probs'][i]:6.1%} " + "#" * int(round(r["probs"][i] * 30)))
        print(f"  saved {out / (Path(p).stem + '_pred.png')}")
    print("\nResearch prototype for a course project - not a medical device.")


if __name__ == "__main__":
    main()
'''
exec(MODEL_DEFS)          # defines make_head, MixedKernelNet, ResNet, DenseNet, MODELS, ARCH_NAMES


# ------------------------------------- locate things --------------------------------------------
def find_art():
    cands = ([Path(ART_DIR)] if ART_DIR else []) + [Path("/kaggle/working/artifacts_v5"), Path("artifacts_v5")]
    root = Path("/kaggle/input")
    if root.is_dir():
        for dp, dn, fn in os.walk(root):
            if len(Path(dp).relative_to(root).parts) > 5:
                dn[:] = []; continue
            if "models_index.json" in fn:
                cands.append(Path(dp))
    for c in cands:
        if (c / "models_index.json").exists():
            return c
    raise FileNotFoundError("artifacts_v5/models_index.json not found. Run this in the same session as the training run, "
                            "or add the run's output as an input, or set ART_DIR.")


def find_data():
    ok = lambda d: all((d / s).is_dir() for s in ("train", "test", "val"))
    for c in ([Path(DATA_DIR)] if DATA_DIR else []) + [Path(DEFAULT_DATA)]:
        if ok(c):
            return c
    root = Path("/kaggle/input")
    for dp, dn, _ in os.walk(root):
        d = Path(dp)
        if len(d.relative_to(root).parts) > 5:
            dn[:] = []; continue
        if ok(d):
            return d
    raise FileNotFoundError("dataset (folder with train/ test/ val/) not found; set DATA_DIR.")


def list_test(base):
    """Same order as training: test, auto_test, val; class 0..4; files sorted."""
    items = []
    for split in ("test", "auto_test", "val"):
        if (base / split).is_dir():
            for c in range(NC):
                for p in sorted(q for q in (base / split / str(c)).iterdir() if q.suffix.lower() in EXT):
                    items.append((p, split, c))
    return items


def load_gray(p):
    return np.asarray(Image.open(p).convert("L").resize((IMG, IMG), Image.BILINEAR), dtype=np.uint8)


# ------------------------------------------ metrics ---------------------------------------------
def metrics_of(y, pred):
    return {"n": int(len(y)), "acc": float(accuracy_score(y, pred)), "f1w": float(f1_score(y, pred, average="weighted")),
            "f1m": float(f1_score(y, pred, average="macro")),
            "kappa": float(cohen_kappa_score(y, pred, weights="quadratic")),
            "balanced_acc": float(recall_score(y, pred, average="macro", labels=list(range(NC)), zero_division=0)),
            "f1_per_class": [float(v) for v in f1_score(y, pred, average=None, labels=list(range(NC)))],
            "cm": confusion_matrix(y, pred, labels=list(range(NC))).tolist()}


def tune_bias(logp_val, y_val, objective):
    def score(b):
        pred = (logp_val + b).argmax(1)
        return (f1_score(y_val, pred, average="macro") if objective == "macro_f1"
                else cohen_kappa_score(y_val, pred, weights="quadratic"))
    b, best = np.zeros(NC), score(np.zeros(NC))
    for _ in range(3):
        for c in range(NC):
            for g in np.arange(-1.5, 1.51, 0.1):
                b2 = b.copy(); b2[c] = g
                s = score(b2)
                if s > best + 1e-9:
                    best, b = s, b2
    return b, best


def show(title, m):
    print(f"\n{'=' * 66}\n{title}\n{'=' * 66}")
    print(f"n={m['n']} | Accuracy {m['acc']*100:.2f}% | weighted F1 {m['f1w']:.4f} | macro F1 {m['f1m']:.4f} | "
          f"kappa {m['kappa']:.4f} | balanced acc {m['balanced_acc']:.4f}")
    print("Per-class F1: " + "  ".join(f"G{i} {f:.3f}" for i, f in enumerate(m["f1_per_class"])))


# ------------------------------------------ plotting --------------------------------------------
def case_title(y, pred, conf):
    return f"true G{y} | pred G{pred} | {conf:.0%}"


def draw_cases(idx, items, y, pred, probs_final, cams, fname, title, rows_cam=True, per_row=None, show_inline=False):
    if len(idx) == 0:                                  # e.g. no mistakes at all
        return
    n = len(idx); per_row = per_row or n
    nr = (n + per_row - 1) // per_row
    fig, axes = plt.subplots(nr * (2 if rows_cam else 1), per_row, figsize=(2.6 * per_row, 2.9 * nr * (2 if rows_cam else 1)),
                             squeeze=False)
    for a in axes.ravel():
        a.axis("off")
    for k, i in enumerate(idx):
        r, c = divmod(k, per_row)
        img = load_gray(items[i][0])
        ax = axes[r * (2 if rows_cam else 1), c]
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(case_title(y[i], pred[i], probs_final[i, pred[i]]), fontsize=8.5,
                     color="#1B5E20" if pred[i] == y[i] else "#B00020")
        if rows_cam and cams is not None:
            ax2 = axes[r * 2 + 1, c]
            ax2.imshow(img, cmap="gray", vmin=0, vmax=255); ax2.imshow(cams[i], cmap="jet", alpha=0.40)
            ax2.set_title("Grad-CAM", fontsize=8)
    fig.suptitle(title, y=1.0, fontsize=11)
    plt.tight_layout(); plt.savefig(OUT_DIR / fname, dpi=140, bbox_inches="tight")
    if show_inline:
        try:
            ip = sys.modules.get("IPython")
            if ip is not None and ip.get_ipython() is not None:
                from IPython.display import display
                display(fig)
        except Exception:
            pass
    plt.close(fig)


# ------------------------------------------- main -----------------------------------------------
def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUT_DIR / "demo_images").mkdir(exist_ok=True); (OUT_DIR / "backup").mkdir(exist_ok=True)
    art = find_art(); base = find_data()
    print(f"artifacts: {art}\ndataset:   {base}\noutput:    {OUT_DIR}")
    meta = json.load(open(art / "models_index.json"))
    tags, kv = meta["tags"], meta["val_kappa"]
    y_val = np.load(art / "val_labels.npy")
    Pv = {t: np.load(art / f"probs_{t}_val.npy") for t in tags}
    Pt = {t: np.load(art / f"probs_{t}_test.npy") for t in tags}
    items = list_test(base)
    y = np.array([c for _, _, c in items], dtype=np.int64)
    splits = np.array([s for _, s, _ in items])
    n_rows = len(next(iter(Pt.values())))
    assert len(items) == n_rows, (f"test file list ({len(items)}) != saved predictions ({n_rows}); an unreadable image was "
                                  "probably skipped during training -- tell Claude")
    official = splits == "test"
    print(f"{len(tags)} trained models | val {len(y_val)} | test rows {n_rows} (official test split: {int(official.sum())})")

    # ---- 1. choose the ensemble on VALIDATION only ----
    def ens(sel, P):
        w = np.clip(np.array([kv[t] for t in sel]), 1e-3, None); w = w / w.sum()
        return sum(wi * P[t] for wi, t in zip(w, sel))
    ranked = sorted(tags, key=lambda t: -kv[t])
    rows = []
    for k in range(1, len(ranked) + 1):
        pv = ens(ranked[:k], Pv).argmax(1)
        rows.append((k, cohen_kappa_score(y_val, pv, weights="quadratic")))
    mk = min(MIN_MODELS, len(ranked))
    best_k = max((r for r in rows if r[0] >= mk), key=lambda r: (round(r[1], 4), r[0]))[0]
    sel = ranked[:best_k]
    print("ranked by val kappa: " + "  ".join(f"{t}:{kv[t]:.3f}" for t in ranked))
    print(f"SELECTED on validation: top-{best_k} -> {sel}")
    Pva, Pte = ens(sel, Pv), ens(sel, Pt)
    bias, vbest = tune_bias(np.log(Pva + 1e-9), y_val, TUNE_OBJECTIVE)
    pred_raw = Pte.argmax(1)
    pred_fin = (np.log(Pte + 1e-9) + bias).argmax(1)
    all_raw = ens(tags, Pt).argmax(1)

    res = {"selected_models": sel, "n_trained": len(tags), "bias": bias.tolist(), "tune_objective": TUNE_OBJECTIVE,
           "val_kappa": {t: kv[t] for t in tags}}
    for name, mask in [("all_test_4008", np.ones(n_rows, bool)), ("official_test", official)]:
        res[name] = {"raw": metrics_of(y[mask], pred_raw[mask]), "tuned": metrics_of(y[mask], pred_fin[mask]),
                     "all_models_raw": metrics_of(y[mask], all_raw[mask])}
        show(f"{name}: selected ensemble RAW", res[name]["raw"])
        show(f"{name}: selected ensemble BIAS-TUNED (final)", res[name]["tuned"])
    json.dump(res, open(OUT_DIR / "results.json", "w"), indent=2)

    # ---- 2. CSV of every test image ----
    import csv
    with open(OUT_DIR / "predictions_all_test.csv", "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["file", "split", "true_grade", "pred_final", "pred_raw", "correct_final"] + [f"p_G{i}" for i in range(NC)])
        Pn = Pte / Pte.sum(1, keepdims=True)
        Pf = np.exp(np.log(Pn + 1e-9) + bias); Pf = Pf / Pf.sum(1, keepdims=True)    # probabilities after the bias
        for i, (p, s, c) in enumerate(items):
            wr.writerow([f"{s}/{c}/{p.name}", s, int(c), int(pred_fin[i]), int(pred_raw[i]), int(pred_fin[i] == c)]
                        + [f"{v:.4f}" for v in Pf[i]])
    print(f"saved predictions_all_test.csv ({len(items)} rows)")

    # ---- 3. model bundle + predict.py ----
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    states = {t: torch.load(art / f"ft_{t}.pt", map_location="cpu")["state"] for t in sel}
    bundle = {"format": 1, "created": datetime.now().isoformat(timespec="seconds"), "class_names": CLASS_NAMES, "img": IMG,
              "bias": [float(v) for v in bias], "tta": "hflip", "note": "from-scratch ensemble, no pretrained weights",
              "models": [{"tag": t, "arch": t[0], "state": states[t], "weight": float(max(kv[t], 1e-3)), "val_kappa": float(kv[t])}
                         for t in sel],
              "metrics": {"official_test": res["official_test"]["tuned"]["acc"], "all_test": res["all_test_4008"]["tuned"]["acc"]}}
    torch.save(bundle, OUT_DIR / "knee_oa_model.pt")
    (OUT_DIR / "predict.py").write_text(PREDICT_SRC.replace("@@" + "MODEL_DEFS@@", MODEL_DEFS))
    print(f"saved knee_oa_model.pt ({(OUT_DIR / 'knee_oa_model.pt').stat().st_size / 2**20:.1f} MB) and predict.py")
    sys.dont_write_bytecode = True
    spec = importlib.util.spec_from_file_location("predict_mod", OUT_DIR / "predict.py")
    pm = importlib.util.module_from_spec(spec); spec.loader.exec_module(pm)
    b = pm.load_bundle(OUT_DIR / "knee_oa_model.pt", dev)

    # ---- 4. choose example images (reproducible) ----
    rng = np.random.RandomState(SEED)
    by_c = {c: list(rng.permutation(np.where(y == c)[0])) for c in range(NC)}
    nb6 = [by_c[c].pop() for c in range(NC)] + [by_c[int(rng.choice([1, 2]))].pop()]     # 6 notebook images: one per grade + 1 extra
    extra5 = [by_c[c].pop() for c in range(NC)]                                           # 5 more, one per grade, all different
    gal25 = [by_c[c].pop() for c in range(NC) for _ in range(5)]
    demo25 = [by_c[c].pop() for c in range(NC) for _ in range(5)]
    conf_fin = Pf[np.arange(n_rows), pred_fin]
    wrong = np.where(pred_fin != y)[0]
    mist = list(wrong[np.argsort(-conf_fin[wrong])][:12])
    cam_idx = [by_c[c].pop() for c in range(NC) for _ in range(2)]

    def cams_for(ids):
        out = {}
        for i in ids:
            x, _ = pm.preprocess(items[i][0], IMG)
            out[i] = pm.grad_cam(b, x, int(pred_fin[i]))
        return out

    # ---- 5. image outputs ----
    cm6 = cams_for(nb6)
    draw_cases(nb6, items, y, pred_fin, Pf, cm6, "notebook_test_images.png",
               "6 test images: true vs predicted (final ensemble) + Grad-CAM", per_row=6, show_inline=True)
    cm5 = cams_for(extra5)
    draw_cases(extra5, items, y, pred_fin, Pf, cm5, "extra_5_test_images.png",
               "5 extra test images (one per grade): true vs predicted + Grad-CAM", per_row=5)
    draw_cases(gal25, items, y, pred_fin, Pf, None, "test_gallery.png",
               "Test gallery: 5 random images per grade (green = correct, red = wrong)", rows_cam=False, per_row=5)
    draw_cases(mist, items, y, pred_fin, Pf, None, "gallery_mistakes.png",
               "Most confident WRONG predictions (neighbouring grades are the usual confusion)", rows_cam=False, per_row=6)
    cmc = cams_for(cam_idx)
    draw_cases(cam_idx, items, y, pred_fin, Pf, cmc, "gradcam_examples.png",
               "Grad-CAM: where the model looks (2 images per grade)", per_row=5)

    # demo images: originals + annotated copies produced by predict.py itself (also a self-test of predict.py)
    agree = 0
    lab = []
    for n, i in enumerate(demo25):
        c = int(y[i]); stem = f"G{c}_{n % 5 + 1:02d}"
        src = items[i][0]
        r = pm.predict_image(b, src, OUT_DIR / "demo_images" / (stem + ".png"), true=c)   # ONE png per image: input | model output | ground truth
        agree += int(r["grade"] == int(pred_fin[i]))
        lab.append((stem + ".png", c, r["grade"], f"{r['confidence']:.3f}"))
    with open(OUT_DIR / "demo_images" / "labels.csv", "w", newline="") as f:
        wr = csv.writer(f); wr.writerow(["file", "true_grade", "predicted_by_predict_py", "confidence"]); wr.writerows(lab)
    print(f"predict.py self-test: agrees with the notebook ensemble on {agree}/{len(demo25)} demo images")

    # ---- 6. metric plots ----
    tun, raw = res["all_test_4008"]["tuned"], res["all_test_4008"]["raw"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
    for ax, m, t in [(axes[0], raw, "raw ensemble"), (axes[1], tun, "bias-tuned (final)")]:
        cm = np.array(m["cm"]); pct = cm / cm.sum(1, keepdims=True).clip(min=1) * 100
        ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
        for i in range(NC):
            for j in range(NC):
                ax.text(j, i, f"{cm[i, j]}\n{pct[i, j]:.0f}%", ha="center", va="center", fontsize=8,
                        color="white" if pct[i, j] > 55 else "black")
        ax.set_xticks(range(NC)); ax.set_yticks(range(NC))
        ax.set_xticklabels([f"G{i}" for i in range(NC)]); ax.set_yticklabels([f"G{i} {CLASS_NAMES[i]}" for i in range(NC)])
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{t}: acc {m['acc']*100:.1f}%  kappa {m['kappa']:.3f}")
    plt.tight_layout(); plt.savefig(OUT_DIR / "confusion_matrix.png", dpi=140, bbox_inches="tight"); plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
    for ax, pr, t in [(axes[0], pred_raw, "raw"), (axes[1], pred_fin, "bias-tuned (final)")]:
        p = precision_score(y, pr, average=None, labels=list(range(NC)), zero_division=0)
        r = recall_score(y, pr, average=None, labels=list(range(NC)), zero_division=0)
        f = f1_score(y, pr, average=None, labels=list(range(NC)), zero_division=0)
        x0, w = np.arange(NC), 0.26
        for off, vals, lab_, col in [(-w, p, "precision", "#4472C4"), (0, r, "recall", "#ED7D31"), (w, f, "F1", "#70AD47")]:
            bars = ax.bar(x0 + off, vals, w, label=lab_, color=col)
            for bb in bars:
                ax.annotate(f"{bb.get_height():.2f}", (bb.get_x() + bb.get_width() / 2, bb.get_height()), xytext=(0, 2),
                            textcoords="offset points", ha="center", fontsize=7)
        ax.set_xticks(x0); ax.set_xticklabels([f"G{i}\n{CLASS_NAMES[i]}" for i in range(NC)]); ax.set_ylim(0, 1.05)
        ax.set_title(f"per-class metrics - {t}"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.savefig(OUT_DIR / "per_class_f1.png", dpi=140, bbox_inches="tight"); plt.close(fig)

    rows_v = dict(V_OLD); rows_v["v5 raw"] = raw; rows_v["v5 final"] = tun
    keys = [("acc", "Accuracy"), ("f1w", "Weighted F1"), ("kappa", "Quadratic kappa")]
    cols = ["#B00020", "#B8860B", "#4472C4", "#8FA9D6", "#5B7DB8", "#2E7D32", "#1B5E20"]
    fig, ax = plt.subplots(figsize=(13, 5.4)); x0, w = np.arange(len(keys)), 0.12
    for i, (nm, m) in enumerate(rows_v.items()):
        bars = ax.bar(x0 + (i - 3) * w, [m[k] for k, _ in keys], w, label=nm, color=cols[i])
        for bb in bars:
            ax.annotate(f"{bb.get_height():.2f}", (bb.get_x() + bb.get_width() / 2, bb.get_height()), xytext=(0, 2),
                        textcoords="offset points", ha="center", fontsize=7)
    ax.set_xticks(x0); ax.set_xticklabels([n for _, n in keys]); ax.set_ylim(0, 1); ax.legend(ncol=7, fontsize=8)
    ax.set_title("All versions (same 4,008-image test set)")
    plt.tight_layout(); plt.savefig(OUT_DIR / "version_comparison.png", dpi=140, bbox_inches="tight"); plt.close(fig)

    nrow = (len(sel) + 3) // 4
    fig, axes = plt.subplots(nrow, 4, figsize=(16, 3.6 * nrow), squeeze=False)
    for a in axes.ravel():
        a.axis("off")
    for ax, t in zip(axes.ravel(), sel):
        hst = json.load(open(art / f"hist_{t}.json")); ax.axis("on")
        ax.plot(hst["train_loss"], label="train loss"); ax.plot(hst["val_loss"], label="val loss")
        a2 = ax.twinx(); a2.plot(hst["val_kappa"], color="purple", ls="--", label="val kappa (EMA)"); a2.set_ylim(0, 1)
        ax.set_title(f"{t} ({ARCH_NAMES[t[0]]})", fontsize=9); ax.set_xlabel("epoch"); ax.legend(loc="upper right", fontsize=6)
    plt.tight_layout(); plt.savefig(OUT_DIR / "training_curves.png", dpi=130, bbox_inches="tight"); plt.close(fig)

    # ---- 7. RESULTS.md ----
    def row(name, m):
        return (f"| {name} | {m['n']} | {m['acc']*100:.2f}% | {m['balanced_acc']:.3f} | {m['f1m']:.3f} | {m['f1w']:.3f} | "
                f"{m['kappa']:.3f} | " + " / ".join(f"{v:.2f}" for v in m["f1_per_class"]) + " |")
    md = [f"# Knee OA KL-grade classification - final results (v5, from scratch)\n",
          f"Generated {datetime.now():%Y-%m-%d %H:%M}. 5 classes (KL 0-4). **No pretrained weights**: random initialisation; the only "
          f"pre-training is self-supervised rotation prediction on our own training images (no labels, no outside data).\n",
          f"Models: {len(tags)} trained (3 architectures x seeds); **{len(sel)} selected on validation kappa**: {', '.join(sel)}. "
          f"Per-class bias tuned on validation ({TUNE_OBJECTIVE}). Nothing was selected or tuned on test.\n",
          "## Data\n", f"- train 4,911 / validation 867 (85/15 split of the 5,778-image train folder)\n"
          f"- test: **{n_rows}** images = test ({int(official.sum())}, the official split) + auto_test + val folders\n",
          "## Results\n", "| set | n | accuracy | balanced acc | macro F1 | weighted F1 | quad. kappa | per-class F1 (G0..G4) |",
          "|---|---|---|---|---|---|---|---|",
          row("all test, raw", raw), row("all test, bias-tuned (final)", tun),
          row("official test, raw", res["official_test"]["raw"]), row("official test, bias-tuned (final)", res["official_test"]["tuned"]),
          row(f"all test, all {len(tags)} models raw", res["all_test_4008"]["all_models_raw"]), "",
          "Previous versions on the same 4,008 images: v1 46.1% / kappa 0.416, v2 31.1% / 0.460, v3 51.0% / 0.579, "
          "v4 54.5% (raw) / 0.625, 55.3% (tuned) / 0.681.\n",
          "## Comparison with published work (numbers as reported by the authors; test sets differ)\n",
          "| work | pretrained? | test | result |", "|---|---|---|---|",
          "| Ensemble of 8 CNNs, Scientific Reports 2023 | yes (ImageNet) | 5-fold CV | 76.9% accuracy |",
          "| EfficientNetB5 baseline, Diagnostics 2025 | yes | official 1,656 | 66.4% accuracy, macro F1 0.602 |",
          "| ResNet-101 / Xception (cited in that paper) | yes | same dataset | 69% / 67.8% |",
          "| from-scratch CNN, GitHub (afzall003) | no | official 1,656 | balanced acc 0.644, kappa 0.753 |",
          f"| **this work (from scratch)** | no | official 1,656 | **{res['official_test']['tuned']['acc']*100:.1f}%** accuracy, "
          f"balanced acc {res['official_test']['tuned']['balanced_acc']:.3f}, kappa {res['official_test']['tuned']['kappa']:.3f} (final); "
          f"raw {res['official_test']['raw']['acc']*100:.1f}% / {res['official_test']['raw']['kappa']:.3f} |", "",
          "The 82% reported in the 2025 CleanLab paper is scored against *relabelled* test labels and is not comparable "
          "(on the original labels the improvement was not significant, McNemar p = 0.076).\n",
          "## Honest notes\n", "- G1 (Doubtful) and G2 (Minimal) remain the hardest grades; most errors are to a neighbouring grade.\n"
          "- Bias tuning trades some accuracy for better F1/kappa on G1-G2; both raw and tuned are reported.\n"
          "- Research prototype, not a medical device.\n",
          "## Files\n", "`knee_oa_model.pt` + `predict.py` (laptop inference), `predictions_all_test.csv`, `results.json`, plots, "
          "`demo_images/`, `backup/all_model_weights.zip`.\n"]
    (OUT_DIR / "RESULTS.md").write_text("\n".join(md))

    # ---- 8. backup of all weights, zip, tidy up ----
    with zipfile.ZipFile(OUT_DIR / "backup" / "all_model_weights.zip", "w", zipfile.ZIP_STORED) as z:
        for t in tags:
            z.write(art / f"ft_{t}.pt", f"ft_{t}.pt")
        z.write(art / "models_index.json", "models_index.json"); z.write(art / "val_labels.npy", "val_labels.npy")
    zpath = shutil.make_archive(str(OUT_DIR.parent / "knee_oa_final"), "zip", OUT_DIR)
    print(f"\nzip: {zpath} ({Path(zpath).stat().st_size / 2**20:.1f} MB)")
    if MOVE_WORK_TO_TEMP and str(art).startswith("/kaggle/working"):
        tmp = Path("/kaggle/temp"); tmp.mkdir(exist_ok=True)
        dest = tmp / "artifacts_v5_work"
        if dest.exists():
            shutil.rmtree(dest)
        shutil.move(str(art), str(dest))
        print(f"moved the {len(list(dest.iterdir()))} working files to {dest} (not part of the notebook output)")
    print("\nFINAL FOLDER:")
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file():
            print(f"  {str(p.relative_to(OUT_DIR)):45s} {p.stat().st_size / 2**20:7.2f} MB")
    print(f"\nHEADLINE (all {n_rows} test images): final {tun['acc']*100:.2f}% acc | macro-F1 {tun['f1m']:.3f} | kappa {tun['kappa']:.3f}"
          f"   raw {raw['acc']*100:.2f}% | kappa {raw['kappa']:.3f}")
    o = res["official_test"]
    print(f"HEADLINE (official {int(official.sum())}-image test): final {o['tuned']['acc']*100:.2f}% | kappa {o['tuned']['kappa']:.3f}"
          f"   raw {o['raw']['acc']*100:.2f}% | kappa {o['raw']['kappa']:.3f}")


main()


In [ ]:
# ===================== FINAL RESULTS (reads /kaggle/working/final) =====================
import json, csv
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

FINAL = Path("/kaggle/working/final") if Path("/kaggle/working/final").exists() else Path("final")
R = json.load(open(FINAL / "results.json"))
CN = ["G0 Healthy", "G1 Doubtful", "G2 Minimal", "G3 Moderate", "G4 Severe"]
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)

print("=" * 78)
print("KNEE OA KL-GRADE CLASSIFICATION  -  FINAL RESULTS  (trained from scratch, no pretrained weights)")
print("=" * 78)
print(f"\nModels trained: {R['n_trained']}   |   selected on VALIDATION: {len(R['selected_models'])} -> {', '.join(R['selected_models'])}")
print(f"Per-class bias (tuned on validation, objective = {R['tune_objective']}): {[round(v, 2) for v in R['bias']]}")
print("Nothing was selected or tuned on the test images.\n")

rows = []
for title, key in [("ALL TEST", "all_test_4008"), ("OFFICIAL TEST", "official_test")]:
    for lab, k in [("raw ensemble", "raw"), ("FINAL (bias-tuned)", "tuned"), (f"all {R['n_trained']} models, raw", "all_models_raw")]:
        m = R[key][k]
        rows.append({"test set": f"{title} (n={m['n']})", "model": lab, "accuracy %": round(m["acc"] * 100, 2),
                     "balanced acc": round(m["balanced_acc"], 3), "macro F1": round(m["f1m"], 3),
                     "weighted F1": round(m["f1w"], 3), "quad. kappa": round(m["kappa"], 3)})
print("-" * 78); print("SUMMARY METRICS"); print("-" * 78)
display(pd.DataFrame(rows))

for title, key in [("ALL TEST", "all_test_4008"), ("OFFICIAL TEST", "official_test")]:
    fin = R[key]["tuned"]
    print("\n" + "-" * 78); print(f"{title} (n={fin['n']}) - FINAL ensemble: per-class F1 and confusion matrix (rows = true, columns = predicted)")
    print("-" * 78)
    f1s = pd.DataFrame({"F1 raw": R[key]["raw"]["f1_per_class"], "F1 final": fin["f1_per_class"]}, index=CN).round(3)
    display(f1s)
    cm = np.array(fin["cm"])
    display(pd.DataFrame(cm, index=[f"true {c}" for c in CN], columns=[f"pred G{i}" for i in range(5)]))
    rec = cm.diagonal() / cm.sum(1).clip(min=1)
    print("recall per grade: " + "   ".join(f"G{i} {r:.0%}" for i, r in enumerate(rec)))

a = R["all_test_4008"]["tuned"]; o = R["official_test"]["tuned"]
print("\n" + "=" * 78)
print(f"HEADLINE  all test (n={a['n']}):      accuracy {a['acc']*100:.2f}%  |  macro-F1 {a['f1m']:.3f}  |  kappa {a['kappa']:.3f}   (raw: {R['all_test_4008']['raw']['acc']*100:.2f}%, kappa {R['all_test_4008']['raw']['kappa']:.3f})")
print(f"HEADLINE  official test (n={o['n']}): accuracy {o['acc']*100:.2f}%  |  macro-F1 {o['f1m']:.3f}  |  kappa {o['kappa']:.3f}   (raw: {R['official_test']['raw']['acc']*100:.2f}%, kappa {R['official_test']['raw']['kappa']:.3f})")
print("=" * 78)

print("\nPUBLISHED WORK FOR COMPARISON (as reported by the authors; test sets differ):")
display(pd.DataFrame([
    ["Ensemble of 8 CNNs, Sci. Reports 2023", "yes (ImageNet)", "5-fold CV", "76.9% accuracy"],
    ["EfficientNetB5 baseline, Diagnostics 2025", "yes", "official 1,656", "66.4% acc, macro-F1 0.602"],
    ["ResNet-101 / Xception (cited there)", "yes", "same dataset", "69% / 67.8%"],
    ["From-scratch CNN, GitHub (afzall003)", "no", "official 1,656", "balanced acc 0.644, kappa 0.753"],
    ["THIS WORK (from scratch)", "no", f"official {o['n']}", f"{o['acc']*100:.1f}% acc, balanced acc {o['balanced_acc']:.3f}, kappa {o['kappa']:.3f}"],
], columns=["work", "pretrained?", "test set", "result"]))

def show(fname, title):
    p = FINAL / fname
    if p.exists():
        print(f"\n### {title}"); display(Image(filename=str(p)))

show("confusion_matrix.png", "Confusion matrices (raw vs final)")
show("per_class_f1.png", "Per-class precision / recall / F1")
show("version_comparison.png", "All versions v1 -> v5")
show("test_gallery.png", "Test gallery: 5 random images per grade (green = right, red = wrong)")
show("extra_5_test_images.png", "5 extra test images, one per grade, with Grad-CAM")
show("gallery_mistakes.png", "Most confident wrong predictions")
show("gradcam_examples.png", "Grad-CAM: where the model looks")
show("training_curves.png", "Training curves of the selected models")

print("\n### Input | Model output | Ground truth  (6 test images, one per grade + 1; the other files are in demo_images/)")
for nm in ["G0_01", "G1_01", "G2_01", "G3_01", "G4_01", "G2_02"]:
    q = FINAL / "demo_images" / (nm + ".png")
    if q.exists():
        display(Image(filename=str(q)))

print("\n### First rows of predictions_all_test.csv (one row per test image)")
display(pd.read_csv(FINAL / "predictions_all_test.csv").head(12))

print("\n### Files in the final folder")
for p in sorted(FINAL.rglob("*")):
    if p.is_file() and "demo_images" not in str(p.parent):
        print(f"  {str(p.relative_to(FINAL)):40s} {p.stat().st_size / 2**20:8.2f} MB")
print(f"  demo_images/  ({len(list((FINAL / 'demo_images').glob('*.png')))} png files)")
z = FINAL.parent / "knee_oa_final.zip"
print(f"\nDOWNLOAD:  {z}  ({z.stat().st_size / 2**20:.1f} MB)  <- right-hand panel > Output > knee_oa_final.zip" if z.exists() else "")
print("On your laptop: put predict.py + knee_oa_model.pt in one folder, then:  python predict.py xray.png")
